# TAU PowerSLURM × Jobflow + Atomate2
### Streamlined, Step-by-Step

**When to use this notebook:** Submit VASP calculations (Static, Relax, DOS, Bands) on TAU PowerSLURM from your local Jupyter, monitor progress, parse results into MongoDB, and visualize structures.

---

### Notebook Workflow

1. **Cell 1 — Config & Initialize**  
   Define cluster login, paths, SLURM resources, VASP modules/command, and MongoDB target (database + collections).  
   Then initialize: open SSH, check remote Python, ensure directories, and write `jobflow_minimal.yaml` to the chosen MongoDB.  
   Also prepares the `make_sbatch` helper used for submissions.

2. **Cell 2 — Workflow Picker & Submit**  
   Interactive panel to choose workflow type, INCAR knobs, k-points, POTCAR set, structure, and SLURM resources.  
   Submits via SLURM and records `JOBID`, run dir, and log paths.

3. **Cell 3 — Monitor**  
   Polls SLURM (`squeue` / `sacct`) until the job reaches a terminal state.  
   Shows job status and tails `.out/.err` logs for quick debugging.

4. **Cell 4 — Parse**  
   Runs pymatgen parsing on `vasprun.xml`/`OUTCAR`/`CONTCAR`.  
   Returns formula, natoms, k-mesh, energy, E-fermi, band gap, etc.  
   Saves a `parse_summary.json` into the run directory.

5. **Cell 5 — Visualize Structure**  
   Fetches latest CONTCAR/POSCAR, applies symmetry, builds a supercell (default 2×2×2), and renders with py3Dmol.  
   Remote results are cached (`.viz_cache`) for speed.

6. **Cell 6 — Band Structure (optional)**  
   If a Bands workflow was run (or a suitable `vasprun.xml` exists), builds a band-structure object, aligns to the Fermi level, and plots.  
   Saves `bandstructure.png` (and optional CSV/JSON) into the run directory; no-ops cleanly if bands are unavailable.

7. **Cell 7 — Close Session**  
   Cleanly closes the persistent SSH/Remote connection.


<h2 id="cell-1">1) Config and Connect</h2>

This cell sets the few things you might change: **username**, **paths** (flows/logs/env/potcars), and **SLURM partition/account** that are then used through the rest of the notebook. 

It creates a config class `cfg = Config()`


In [1]:
# --- Atomate2 / Jobflow-Remote Setup UI (status at bottom) ---
import os, json, threading, getpass, select, socketserver, time
import ipywidgets as widgets
from IPython.display import display, clear_output

# Dependencies
try:
    import paramiko
except ImportError as e:
    raise RuntimeError("Paramiko must be installed (e.g., `pip install paramiko`).") from e

PROFILE_PATH = os.path.expanduser("~/.atomate2_remote_ui.json")

DEFAULTS = dict(
    remote_host="powerslurm-login.tau.ac.il",
    username="leeburton",
    port=22,
    key_file="",
    password="",
    keepalive_s=30,
    open_mongo_tunnel=True,
    mongo_remote_host="132.66.112.243",
    mongo_remote_port=27017,
    mongo_local_port=27017,
    VASP_CMD="mpirun -n $SLURM_NTASKS vasp_std",
    JOBFLOW_CONFIG_FILE="/bmd-db/lee/jobflow_minimal.yaml",
    PMG_VASP_PSP_DIR="/bmd-db/lee/potcars",
    remote_env_dir="/bmd/lee/envs/atomate2_remote",
    flows_dir="/bmd-db/lee/flows",
    logs_dir="/bmd-db/lee/logs",
)

SSH_SESSION = {"client": None, "tunnels": [], "connected": False}

# --- Status + spinner ---
status_html = widgets.HTML("<b style='color:red'>🔴 Not connected</b>")
spinner = widgets.HTML("")
def show_spinner(on=True, msg=""):
    spinner.value = "⏳ " + msg if on else ""

# --- UI widgets ---
w_host = widgets.Text(value=DEFAULTS["remote_host"], description="Host", layout=widgets.Layout(width="50%"))
w_user = widgets.Text(value=DEFAULTS["username"], description="Username", layout=widgets.Layout(width="40%"))
w_port = widgets.IntText(value=DEFAULTS["port"], description="Port", layout=widgets.Layout(width="20%"))

w_key  = widgets.Text(value=DEFAULTS["key_file"], description="Key file", placeholder="~/.ssh/id_rsa (optional)", layout=widgets.Layout(width="60%"))
w_pass = widgets.Password(value=DEFAULTS["password"], description="Password", placeholder="prefer key/agent", layout=widgets.Layout(width="40%"))
w_keep = widgets.IntSlider(value=DEFAULTS["keepalive_s"], min=0, max=300, step=5, description="Keepalive (s)", readout=True)

w_tunnel_on   = widgets.Checkbox(value=DEFAULTS["open_mongo_tunnel"], description="Open Mongo tunnel")
w_mongo_host  = widgets.Text(value=DEFAULTS["mongo_remote_host"], description="Mongo host")
w_mongo_rport = widgets.IntText(value=DEFAULTS["mongo_remote_port"], description="Mongo rport")
w_mongo_lport = widgets.IntText(value=DEFAULTS["mongo_local_port"], description="Mongo lport")

# --- Environment widgets (with wider labels so they don't truncate) ---
w_vasp   = widgets.Text(value=DEFAULTS["VASP_CMD"], description="VASP_CMD")
w_jobflow= widgets.Text(value=DEFAULTS["JOBFLOW_CONFIG_FILE"], description="JOBFLOW CFG")
w_psp    = widgets.Text(value=DEFAULTS["PMG_VASP_PSP_DIR"], description="POTCAR dir")
w_env    = widgets.Text(value=DEFAULTS["remote_env_dir"], description="Remote env dir")
for w in (w_vasp, w_jobflow, w_psp, w_env):
    w.layout = widgets.Layout(width="100%")  # field spans row
    w.style = {"description_width": "170px"} # wider label space

w_flows = widgets.Text(value=DEFAULTS["flows_dir"], description="flows dir")
w_logs  = widgets.Text(value=DEFAULTS["logs_dir"], description="logs dir")

btn_connect    = widgets.Button(description="Connect", button_style="success", icon="plug")
btn_disconnect = widgets.Button(description="Disconnect", button_style="warning", icon="power-off", disabled=True)
btn_check      = widgets.Button(description="Check remote env", icon="check", tooltip="Verify python, modules, and vasp_std")
btn_save       = widgets.Button(description="Save profile", icon="save")
btn_load       = widgets.Button(description="Load profile", icon="folder-open")

out = widgets.Output(layout={"border": "1px solid #ddd"})

# --- Helpers (port forwarding, etc.) ---
class ForwardServer(socketserver.ThreadingTCPServer):
    daemon_threads = True
    allow_reuse_address = True

class Handler(socketserver.BaseRequestHandler):
    def handle(self):
        try:
            chan = self.ssh_transport.open_channel(
                "direct-tcpip", (self.chain_host, self.chain_port), self.request.getpeername()
            )
        except Exception:
            return
        if chan is None:
            return
        while True:
            r, _, _ = select.select([self.request, chan], [], [])
            if self.request in r:
                data = self.request.recv(1024)
                if not data:
                    break
                chan.send(data)
            if chan in r:
                data = chan.recv(1024)
                if not data:
                    break
                self.request.send(data)
        try:
            chan.close()
        except Exception:
            pass
        try:
            self.request.close()
        except Exception:
            pass

def forward_tunnel(local_port, remote_host, remote_port, transport, stop_event):
    class SubHandler(Handler):
        chain_host = remote_host
        chain_port = remote_port
        ssh_transport = transport
    server = ForwardServer(("", local_port), SubHandler)
    def loop():
        with server:
            while not stop_event.is_set():
                server.handle_request()
    t = threading.Thread(target=loop, daemon=True)
    t.start()
    return server, t

def _expanduser(path): 
    return os.path.expanduser(path) if path else ""

def _set_env_from_widgets():
    os.environ["VASP_CMD"] = w_vasp.value
    os.environ["JOBFLOW_CONFIG_FILE"] = w_jobflow.value
    os.environ["PMG_VASP_PSP_DIR"] = w_psp.value
    os.environ["ATOMATE2_REMOTE_ENV"] = w_env.value
    os.environ["CUSTODIAN_NO_GZIP"] = "1"
    os.environ["ATOMATE2_VASP_ZIP_FILES"] = "False"

def _toggle_mongo_inputs(enabled: bool):
    for w in (w_mongo_host, w_mongo_rport, w_mongo_lport):
        w.disabled = not enabled

def _toggle_controls(connected: bool, busy: bool = False):
    btn_connect.disabled = connected or busy
    btn_disconnect.disabled = (not connected) or busy
    for w in (w_host, w_user, w_port, w_key, w_pass, w_keep,
              w_tunnel_on, w_mongo_host, w_mongo_rport, w_mongo_lport,
              w_vasp, w_jobflow, w_psp, w_env, w_flows, w_logs):
        w.disabled = connected or busy
    btn_check.disabled = not connected or busy
    btn_save.disabled = busy
    btn_load.disabled = busy

def _profile_from_widgets():
    d = {
        "remote_host": w_host.value.strip(),
        "username": w_user.value.strip(),
        "port": int(w_port.value),
        "key_file": w_key.value.strip(),
        "password": "",  # never persist password
        "keepalive_s": int(w_keep.value),
        "open_mongo_tunnel": bool(w_tunnel_on.value),
        "mongo_remote_host": w_mongo_host.value.strip(),
        "mongo_remote_port": int(w_mongo_rport.value),
        "mongo_local_port": int(w_mongo_lport.value),
        "VASP_CMD": w_vasp.value,
        "JOBFLOW_CONFIG_FILE": w_jobflow.value,
        "PMG_VASP_PSP_DIR": w_psp.value,
        "remote_env_dir": w_env.value,
        "flows_dir": w_flows.value,
        "logs_dir": w_logs.value,
    }
    return d

def _apply_profile(d: dict):
    w_host.value = d.get("remote_host", w_host.value)
    w_user.value = d.get("username", w_user.value)
    w_port.value = d.get("port", w_port.value)
    w_key.value  = d.get("key_file", w_key.value)
    # never set password back
    w_keep.value = d.get("keepalive_s", w_keep.value)
    w_tunnel_on.value = d.get("open_mongo_tunnel", w_tunnel_on.value)
    w_mongo_host.value = d.get("mongo_remote_host", w_mongo_host.value)
    w_mongo_rport.value = d.get("mongo_remote_port", w_mongo_rport.value)
    w_mongo_lport.value = d.get("mongo_local_port", w_mongo_lport.value)
    w_vasp.value = d.get("VASP_CMD", w_vasp.value)
    w_jobflow.value = d.get("JOBFLOW_CONFIG_FILE", w_jobflow.value)
    w_psp.value = d.get("PMG_VASP_PSP_DIR", w_psp.value)
    w_env.value = d.get("remote_env_dir", w_env.value)
    w_flows.value = d.get("flows_dir", w_flows.value)
    w_logs.value = d.get("logs_dir", w_logs.value)
    _toggle_mongo_inputs(bool(w_tunnel_on.value))

def _load_profile_from_disk():
    if os.path.exists(PROFILE_PATH):
        try:
            with open(PROFILE_PATH, "r") as f:
                return json.load(f)
        except Exception:
            return None

def _save_profile_to_disk(d: dict):
    try:
        with open(PROFILE_PATH, "w") as f:
            json.dump(d, f, indent=2)
        return True
    except Exception:
        return False

def _close_all_tunnels():
    for tinfo in SSH_SESSION.get("tunnels", []):
        try:
            tinfo["stop_event"].set()
        except Exception:
            pass
        try:
            tinfo["server"].server_close()
        except Exception:
            pass
    SSH_SESSION["tunnels"] = []

def _disconnect():
    _close_all_tunnels()
    try:
        if SSH_SESSION.get("client"):
            SSH_SESSION["client"].close()
    except Exception:
        pass
    SSH_SESSION["client"] = None
    SSH_SESSION["connected"] = False

def _update_status_ok(host, user, suffix=""):
    status_html.value = f"<b style='color:green'>🟢 Connected to {host} as {user}{suffix}</b>"

def _update_status_err(msg):
    status_html.value = f"<b style='color:darkorange'>⚠️ {msg}</b>"

def establish_connection(_):
    import posixpath, shlex, textwrap
    out.clear_output()
    show_spinner(True, "Connecting…")
    _toggle_controls(connected=False, busy=True)
    with out:
        print("Starting SSH connection...")
        host, user, port = w_host.value.strip(), w_user.value.strip(), int(w_port.value)
        keyfile = _expanduser(w_key.value.strip())
        password = w_pass.value or None
        keepalive = int(w_keep.value)

        client = paramiko.SSHClient()
        client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

        try:
            if keyfile:
                pkey = None
                errs = []
                for KeyClass in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
                    try:
                        pkey = KeyClass.from_private_key_file(keyfile, password=password)
                        break
                    except Exception as e:
                        errs.append(str(e))
                        continue
                if pkey is None:
                    raise RuntimeError("Could not read private key. Tried RSA/ECDSA/Ed25519.\n" + "\n".join(errs))
                client.connect(hostname=host, port=port, username=user, pkey=pkey,
                               allow_agent=True, look_for_keys=False, timeout=20)
            else:
                client.connect(hostname=host, port=port, username=user, password=password,
                               allow_agent=True, look_for_keys=True, timeout=20)

            transport = client.get_transport()
            if not (transport and transport.is_active()):
                raise RuntimeError("SSH transport failed to start.")

            if keepalive > 0:
                transport.set_keepalive(keepalive)

            SSH_SESSION["client"] = client
            SSH_SESSION["connected"] = True
            _set_env_from_widgets()
            globals()["_ATOMATE2_PATHS"] = {"flows_dir": w_flows.value, "logs_dir": w_logs.value}

            # Optional MongoDB tunnel
            if w_tunnel_on.value:
                lport = int(w_mongo_lport.value); rhost = w_mongo_host.value.strip(); rport = int(w_mongo_rport.value)
                stop_event = threading.Event()
                server, thread = forward_tunnel(lport, rhost, rport, transport, stop_event)
                SSH_SESSION["tunnels"].append({
                    "server": server, "thread": thread, "stop_event": stop_event,
                    "local_port": lport, "remote": (rhost, rport)
                })
                print(f"MongoDB tunnel open: localhost:{lport} -> {rhost}:{rport}")

            # Derive helpers
            _paths = globals().get("_ATOMATE2_PATHS") or {}
            _flows_dir = _paths.get("flows_dir") or DEFAULTS["flows_dir"]
            _logs_dir  = _paths.get("logs_dir")  or DEFAULTS["logs_dir"]
            _potcars_dir = os.environ.get("PMG_VASP_PSP_DIR") or DEFAULTS["PMG_VASP_PSP_DIR"]
            _remote_env_dir = os.environ.get("ATOMATE2_REMOTE_ENV") or DEFAULTS["remote_env_dir"]

            try:
                _username = transport.get_username() if transport else (getpass.getuser() or DEFAULTS["username"])
            except Exception:
                _username = DEFAULTS["username"]

            from dataclasses import dataclass
            @dataclass
            class Cfg:
                username: str
                flows_dir: str
                logs_dir: str
                potcars_dir: str
                remote_env_dir: str
                partition: str | None = None
                account: str | None = None

            def _env_bin(cfg: "Cfg") -> str:
                p = (cfg.remote_env_dir or "").rstrip("/")
                return (p + "/bin") if p else "/usr/bin"

            class Remote:
                def __init__(self, client):
                    self.client = client
                def run(self, cmd: str, check: bool = False, modules: bool = False, export_env: bool = False):
                    parts = ["set -e -o pipefail"]
                    if export_env:
                        for k in ("VASP_CMD","JOBFLOW_CONFIG_FILE","PMG_VASP_PSP_DIR"):
                            v = os.environ.get(k)
                            if v:
                                parts.append(f"export {k}={shlex.quote(v)}")
                    if modules:
                        parts += [
                            "module purge >/dev/null 2>&1 || true",
                            "module load intel/rocky8-oneAPI-2023 >/dev/null 2>&1 || true",
                            "module load vasp/rocky8-intel-6.4.1  >/dev/null 2>&1 || true",
                        ]
                    parts.append(cmd)
                    full = "\n".join(parts)
                    stdin, stdout, stderr = self.client.exec_command(full, get_pty=False)
                    out = stdout.read().decode("utf-8", "ignore")
                    err = stderr.read().decode("utf-8", "ignore")
                    rc  = stdout.channel.recv_exit_status()
                    if check and rc != 0:
                        raise RuntimeError(err or out or f"Remote rc={rc}")
                    return rc, out, err

                def put_text(self, remote_path: str, text: str, mode: int = 0o640):
                    sftp = self.client.open_sftp()
                    try:
                        parent = posixpath.dirname(remote_path.rstrip("/"))
                        try:
                            sftp.stat(parent)
                        except IOError:
                            self.run(f"mkdir -p {shlex.quote(parent)}", check=False)
                        with sftp.file(remote_path, "w") as f:
                            f.write(text)
                        sftp.chmod(remote_path, mode)
                    finally:
                        sftp.close()

            def _ensure_live_remote():
                t = client.get_transport()
                if not (t and t.is_active()):
                    raise RuntimeError("SSH transport is not active. Re-run the connection cell.")
                return rmt

            def make_sbatch(cfg: Cfg, job_body: str) -> str:
                exports = []
                for k in ("VASP_CMD","JOBFLOW_CONFIG_FILE","PMG_VASP_PSP_DIR","CUSTODIAN_NO_GZIP","ATOMATE2_VASP_ZIP_FILES"):
                    v = os.environ.get(k)
                    if v is not None:
                        exports.append(f"export {k}={shlex.quote(v)}")
                return f"""#!/usr/bin/env bash
set -e -o pipefail

# -- quiet module loads (compute node) --
module purge >/dev/null 2>&1 || true
module load intel/rocky8-oneAPI-2023 >/dev/null 2>&1 || true
module load vasp/rocky8-intel-6.4.1  >/dev/null 2>&1 || true

# -- reasonable stack size for VASP --
ulimit -s 81920 || true

# -- propagate environment expected by the runner --
{os.linesep.join(exports)}

# -- POTCAR sanity (warn and show layout) --
echo "PMG_VASP_PSP_DIR=$PMG_VASP_PSP_DIR"
ls -ld "$PMG_VASP_PSP_DIR"/POT_PAW_* >/dev/null 2>&1 || echo "⚠️ No POT_PAW_* dir found under $PMG_VASP_PSP_DIR"
ls -l "$PMG_VASP_PSP_DIR/POT_PAW_PBE_64/Si" >/dev/null 2>&1 || echo "⚠️ Si POTCAR not visible at $PMG_VASP_PSP_DIR/POT_PAW_PBE_64/Si"

# -- user-provided body (writes run_job.py and runs {_env_bin(cfg)}/python) --
{job_body.strip()}
"""

            cfg = Cfg(
                username=_username,
                flows_dir=_flows_dir,
                logs_dir=_logs_dir,
                potcars_dir=_potcars_dir,
                remote_env_dir=_remote_env_dir,
                partition=None,
                account=None,
            )
            rmt = Remote(client)
            globals().update(
                cfg=cfg, rmt=rmt,
                _ensure_live_remote=_ensure_live_remote, make_sbatch=make_sbatch
            )

            # remote prep/sanity
            try:
                rc, out_mk, _ = rmt.run(
                    f'mkdir -p {shlex.quote(cfg.flows_dir)} {shlex.quote(cfg.logs_dir)} && echo "[remote] ensured flows/logs dirs"',
                    check=False, modules=False, export_env=False
                )
                print((out_mk or "").strip() or "[remote] flows/logs prepared")
            except Exception as _e:
                print(f"[warn] could not prepare remote dirs: { _e }")

            try:
                pybin = (cfg.remote_env_dir.rstrip("/") + "/bin/python") if cfg.remote_env_dir else "/usr/bin/python"
                rc, out_py, err_py = rmt.run(
                    f'command -v {shlex.quote(pybin)} >/dev/null && {shlex.quote(pybin)} --version || echo "[note] {pybin} not found"',
                    check=False, modules=False, export_env=False
                )
                msg = (out_py or err_py).strip()
                if msg:
                    print(msg)
            except Exception:
                pass

            _update_status_ok(host, user, " — helpers ready")
            _toggle_controls(connected=True, busy=False)
        except Exception as e:
            _disconnect()
            _update_status_err(f"Connection error: {e}")
            print("Connection error:", repr(e))
            _toggle_controls(connected=False, busy=False)
        finally:
            show_spinner(False)

def do_disconnect(_):
    out.clear_output()
    show_spinner(True, "Disconnecting…")
    _toggle_controls(connected=True, busy=True)
    with out:
        try:
            _close_all_tunnels()
            if SSH_SESSION.get("client"):
                SSH_SESSION["client"].close()
            print("Disconnected.")
        except Exception as e:
            print("While disconnecting:", e)
    _disconnect()
    status_html.value = "<b style='color:red'>🔴 Not connected</b>"
    _toggle_controls(connected=False, busy=False)
    show_spinner(False)

def do_check(_):
    if not SSH_SESSION.get("connected"):
        _update_status_err("Not connected")
        return
    out.clear_output()
    with out:
        print("Running remote checks…")
        try:
            # Check python
            pybin = (cfg.remote_env_dir.rstrip("/") + "/bin/python") if cfg.remote_env_dir else "/usr/bin/python"
            rc, out_py, err_py = rmt.run(f'command -v {pybin} && {pybin} --version', check=True)
            print(out_py or err_py)

            # Check module loads and vasp_std
            rc, out_md, err_md = rmt.run('module purge >/dev/null 2>&1 || true && '
                                         'module load intel/rocky8-oneAPI-2023 >/dev/null 2>&1 || true && '
                                         'module load vasp/rocky8-intel-6.4.1  >/dev/null 2>&1 || true && '
                                         'command -v vasp_std || which vasp_std || echo "vasp_std not on PATH"', check=False)
            print((out_md or err_md).strip())

            # Echo env vars important for later cells
            rc, out_env, _ = rmt.run('echo "VASP_CMD=$VASP_CMD"; '
                                     'echo "JOBFLOW_CONFIG_FILE=$JOBFLOW_CONFIG_FILE"; '
                                     'echo "PMG_VASP_PSP_DIR=$PMG_VASP_PSP_DIR"', check=False, export_env=True)
            print(out_env.strip())
        except Exception as e:
            print("Check failed:", e)

def on_tunnel_toggle(change):
    _toggle_mongo_inputs(change["new"])

def do_save(_):
    d = _profile_from_widgets()
    ok = _save_profile_to_disk(d)
    with out:
        print("Profile saved to", PROFILE_PATH if ok else "(failed to save)")

def do_load(_):
    prof = _load_profile_from_disk()
    with out:
        if prof:
            _apply_profile(prof)
            print("Profile loaded from", PROFILE_PATH)
        else:
            print("No profile found at", PROFILE_PATH)

# wire events
w_tunnel_on.observe(on_tunnel_toggle, names="value")
_toggle_mongo_inputs(bool(w_tunnel_on.value))

btn_connect.on_click(establish_connection)
btn_disconnect.on_click(do_disconnect)
btn_check.on_click(do_check)
btn_save.on_click(do_save)
btn_load.on_click(do_load)

# layout: accordions to declutter
ssh_box = widgets.VBox([widgets.HBox([w_host, w_user, w_port]),
                        widgets.HBox([w_key, w_pass]),
                        w_keep])
mongo_box = widgets.HBox([w_tunnel_on, w_mongo_host, w_mongo_rport, w_mongo_lport])
env_box = widgets.VBox([w_vasp, w_jobflow, w_psp, w_env])
paths_box = widgets.HBox([w_flows, w_logs])

acc = widgets.Accordion(children=[ssh_box, mongo_box, env_box, paths_box])
for i, title in enumerate(["SSH", "Mongo Tunnel (optional)", "Environment", "Paths"]):
    acc.set_title(i, title)

buttons = widgets.HBox([btn_connect, btn_disconnect, btn_check, btn_save, btn_load, spinner])

ui = widgets.VBox([
    widgets.HTML("<h3>Atomate2 / Jobflow-Remote Connection & Defaults</h3>"),
    acc,
    buttons,
    status_html,
    out
])
display(ui)


<a id="cell-3"></a>

## 2) Build Job & Submit

Submits a **minimal Si static** job to SLURM.

- Ensures POTCAR links, writes an `sbatch` script, and **submits** in one go.
- Sets `VASP_CMD`, `PMG_VASP_PSP_DIR`, and `JOBFLOW_CONFIG_FILE` **inside** the batch script.
- Creates timestamped run/log files and prints **JOBID**, **RUN_DIR**, and **LOGS**.

**What you might change:** replace the example `job_python` with your own workflow (or just keep it).  
Everything else can stay as-is.


In [2]:
# --- Cell 2: VASP Workflow Picker + Builder + Submit (job-state persistence) ---
# Requires from Cell 1: cfg, rmt, _ensure_live_remote, make_sbatch, _env_bin(cfg)
import os, re, time, json, posixpath, shlex, ipywidgets as w
from IPython.display import display, Markdown, HTML
from pathlib import Path
import datetime as dt

# ==== persistence helpers ====
LOCAL_STATE_DIR = Path.home() / ".atomate2_jobs"
LOCAL_STATE_DIR.mkdir(parents=True, exist_ok=True)

def _now_str():
    return dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def _safe_getattr(obj, name, default=None):
    try:
        return getattr(obj, name, default)
    except Exception:
        return default

def _write_remote_text(remote_path: str, text: str):
    R = globals().get("rmt")
    if hasattr(R, "put_text"):
        try:
            R.put_text(remote_path, text)
            return True
        except Exception:
            pass
    try:
        tmp = posixpath.join("/tmp", f"jobstate_{int(time.time())}_{os.getpid()}.json")
        payload = text.replace("\\", "\\\\").replace("$", r"\$").replace("`", r"\`")
        cmd = f"set -e; cat > {shlex.quote(tmp)} <<'JSON'\n{payload}\nJSON\nmv -f {shlex.quote(tmp)} {shlex.quote(remote_path)}"
        rc, _, _ = R.run(cmd, check=False, modules=False, export_env=False)
        return rc == 0
    except Exception:
        return False

def save_job_state(job_id: str, run_dir: str, log_paths: dict, extra: dict | None = None):
    cfg_obj = globals().get("cfg")
    state = {
        "job_id": str(job_id),
        "run_dir": str(run_dir),
        "log_paths": dict(log_paths or {}),
        "saved_at": _now_str(),
        "config": {
            "username": _safe_getattr(cfg_obj, "username", None),
            "cluster_logs_root": _safe_getattr(cfg_obj, "logs_dir", None),
            "cluster_flows_root": _safe_getattr(cfg_obj, "flows_dir", None),
            "remote_env_dir": _safe_getattr(cfg_obj, "remote_env_dir", None),
        },
    }
    if extra:
        state.update(extra)
    f_local = LOCAL_STATE_DIR / f"{job_id}.json"
    f_local.write_text(json.dumps(state, indent=2))
    print(f"[saved] {f_local}")
    logs_root = _safe_getattr(cfg_obj, "logs_dir", None)
    if logs_root:
        remote_json = posixpath.join(logs_root.rstrip("/"), f"job_{job_id}.json")
        ok = _write_remote_text(remote_json, json.dumps(state, indent=2))
        print(f"[saved] {remote_json} on cluster" if ok else f"[warn] could not save remote copy: {remote_json}")

# ---- tiny utils ----
def _sanitize_label(s: str) -> str:
    import re as _re
    s = (s or "").strip()
    s = _re.sub(r"\s+", "-", s)
    s = _re.sub(r"[^A-Za-z0-9._-]+", "", s)
    return (s or "vasp_run")[:60].rstrip("-_.") or "vasp_run"

def _coerce_int(x, default):
    try:
        return int(x)
    except Exception:
        return int(default)

_ensure_rmt = globals().get("_ensure_live_remote") or (lambda: rmt)
_env_bin = globals().get("_env_bin") or (lambda cfg: (getattr(cfg, "remote_env_dir","") or "").rstrip("/") + "/bin")

def _inject_header(script_text: str, header: str) -> str:
    lines = script_text.splitlines()
    return ("\n".join([lines[0], header.rstrip("\n")] + lines[1:]) + "\n") if (lines and lines[0].startswith("#!")) \
        else (header.rstrip("\n") + "\n" + script_text.rstrip("\n") + "\n")

# ---- submit helper (runner executes all logic on compute node) ----
def fast_submit_from_spec(flow_spec, label, ntasks, mem_gb, wall, nodes=1, dry_run=False, mp_api_key: str | None = None):
    r = _ensure_rmt()
    ts = time.strftime("%Y%m%d-%H%M%S")
    label = _sanitize_label(label)
    run_name = f"{label}-{ts}"
    run_dir  = posixpath.join(cfg.flows_dir, run_name)
    log_out  = posixpath.join(cfg.logs_dir, f"{run_name}.out")
    log_err  = posixpath.join(cfg.logs_dir, f"{run_name}.err")
    spec_json = json.dumps(flow_spec)

    pot_func   = flow_spec.get("potcar_functional", "PBE_64")
    pot_target = posixpath.join(cfg.potcars_dir, pot_func)

    if flow_spec.get("structure", {}).get("type") == "path":
        spath = flow_spec["structure"].get("path", "")
        rc, _, _ = r.run(f'test -f "{spath}" || test -d "{spath}"', check=False, modules=False, export_env=False)
        if rc != 0:
            raise FileNotFoundError(f"Remote structure path not found: {spath}")

    # ----- compact runner (helperized) -----
    job_python = r'''
import os, sys, json, glob, shutil, warnings, gzip, subprocess, re, traceback, time, shlex
from pathlib import Path

print("[runner] python:", sys.version.replace("\n"," "))
def _pkg_ver(name):
    try:
        m=__import__(name)
        v=getattr(m,"__version__", "<no __version__>")
        print(f"[runner] {name} version:", v)
    except Exception as e:
        print(f"[runner] {name} import failed:", e)
for _p in ("atomate2","jobflow","pymatgen","custodian"):
    _pkg_ver(_p)

spec = json.loads(__SPEC__)
wf = (spec.get("workflow") or "static").lower()

def log(msg, *, end="\n"):
    print("[runner] " + str(msg), file=sys.stderr, end=end, flush=True)

# ----- ENCUT policy -----
ENCUT_STATIC_PREP_DEFAULT = 520
ENCUT_RELAX_DEFAULT       = 580
ENCUT_STATIC_FINAL_DEFAULT= 620

def _build_vasp_cmd():
    """
    Build a robust VASP launcher.
    - If user provided VASP_CMD, use it.
    - If it begins with mpirun under Slurm, auto-swap to srun so PMI/PMIx is wired.
    """
    v = os.environ.get("VASP_CMD") or "srun --mpi=pmi2 -n ${SLURM_NTASKS:-1} vasp_std"
    v = os.path.expandvars(v).strip()
    try:
        tokens = shlex.split(v)
        if tokens and tokens[0] == "mpirun" and os.environ.get("SLURM_JOB_ID"):
            ntasks = os.environ.get("SLURM_NTASKS", "1")
            v = f"srun --mpi=pmi2 -n {ntasks} vasp_std"
            tokens = shlex.split(v)
        return tokens
    except Exception:
        return v.split()

# quiet noisy warnings
try:
    from pymatgen.io.vasp.sets import BadInputSetWarning
    warnings.filterwarnings("ignore", category=BadInputSetWarning)
except Exception:
    pass

def latest(patterns, root="."):
    c=[]
    for p in patterns:
        c.extend(glob.glob(os.path.join(root, p), recursive=True))
    return max(c, key=lambda p: os.path.getmtime(p)) if c else None

def copy_keys(src, dst):
    os.makedirs(dst, exist_ok=True)
    keys=("INCAR","KPOINTS","POSCAR","CONTCAR","OUTCAR","vasprun.xml","OSZICAR","EIGENVAL","DOSCAR","PROCAR",
          "CHG","CHGCAR","WAVECAR","PCDAT","REPORT","IBZKPT","XDATCAR","custodian.json","vasp.out","std_err.txt")
    for f in keys:
        found=None
        for g in (f, f+".gz"):
            p=os.path.join(src, g)
            if os.path.exists(p):
                found=p; break
        if found:
            try:
                dstp=os.path.join(dst, os.path.basename(found))
                try:
                    same = os.path.exists(dstp) and os.path.samefile(found, dstp)
                except Exception:
                    same = (os.path.abspath(found) == os.path.abspath(dstp))
                if not same:
                    shutil.copy2(found, dstp)
            except Exception:
                pass

def _is_hse_incar(d):
    if not isinstance(d, dict): return False
    v=d.get("LHFCALC")
    if isinstance(v, str) and v.strip().strip(".").upper() in ("T","TRUE","YES"): return True
    if v is True: return True
    for k in ("AEXX","HFSCREEN","ALDAC","LHFCALC_HYBRID"):
        if k in d: return True
    gga=d.get("GGA")
    if isinstance(gga, str) and "HSE" in gga.upper(): return True
    return False

def incar_static(u, allow_ncore=True):
    u=dict(u or {})
    for k in ("GGA","ENAUG","LMIXTAU"): u.setdefault(k, None)
    u.setdefault("ISMEAR",-5)
    if u.get("ISMEAR",-5)==-5: u.setdefault("SIGMA", None)
    u.setdefault("EDIFF",1e-6)
    u.setdefault("ALGO","Normal")
    for k,v in {"NEDOS":3001,"LORBIT":11,"LVTOT":True,"LAECHG":True,"LCHARG":True,"LWAVE":True,"LELF":True}.items():
        u.setdefault(k,v)
    if allow_ncore and not _is_hse_incar(u):
        u.setdefault("NCORE", 2)
    return u

def incar_relax(u, user=None):
    """
    INCAR for relax steps.
    Defaults:
      - LCHARG=False, LWAVE=False  (save I/O)
      - ADDGRID=True               (force/stress smoothing)
      - EDIFFG=-0.01               (target forces ~0.01 eV/Å)
      - HSE06 relax only: PRECFOCK=Fast, ALGO=Damped
    """
    u = dict(u or {})
    o = dict(user or {})

    # Outputs trimmed unless user explicitly asked otherwise
    if "LCHARG" not in o: u["LCHARG"] = False
    if "LWAVE"  not in o: u["LWAVE"]  = False
    for k in ("LAECHG","LVTOT","LELF","LVHAR","LORBIT"):
        if k not in o: u[k] = None

    # Relax defaults
    for k in ("GGA","ENAUG","LMIXTAU"): u.setdefault(k, None)
    u.setdefault("ALGO","Fast")
    u.setdefault("ADDGRID", True)
    u.setdefault("EDIFFG", -0.01)

    # Hybrid-only tweaks for RELAX (requested): PRECFOCK=Fast (+ ensure ALGO Damped)
    if _is_hse_incar(u) or _is_hse_incar(o):
        u.setdefault("PRECFOCK", "Fast")
        u.setdefault("ALGO", "Damped")  # prefer Damped for HSE stability

    # Parallel defaults for PBE (avoid for HSE)
    if not _is_hse_incar(u) and not _is_hse_incar(o):
        u.setdefault("NCORE", 2)

    return u

def ksettings(structure, k_conf):
    if not k_conf: return None
    mode=k_conf.get("mode"); val=k_conf.get("value")
    if mode=="mesh":
        from pymatgen.io.vasp.inputs import Kpoints
        nx,ny,nz=(int(val[0]),int(val[1]),int(val[2]))
        nat=len(structure)
        def mesh_for(kppa):
            kp=Kpoints.automatic_density(structure,int(max(1,kppa)))
            if kp.kpts:
                m=kp.kpts[0]
                if isinstance(m,(list,tuple)) and len(m)>=3:
                    return (int(m[0]),int(m[1]),int(m[2]))
            return (0,0,0)
        target=(nx,ny,nz)
        kppa=max(1,nx*ny*nz*max(1,nat))
        seen=set(); found=None
        for _ in range(64):
            m=mesh_for(kppa)
            if m==target: found=kppa; break
            if m in seen: break
            seen.add(m)
            pt=nx*ny*nz
            pm=max(1,m[0]*m[1]*m[2])
            scale=max(pt/pm, nx/max(1,m[0]), ny/max(1,m[1]), nz/max(1,m[2]))
            kppa=int(max(1, kppa*(1.25 if scale<1 else min(3.0, 1.15*scale))))
        if found is None:
            k0=max(1,nx*ny*nz*max(1,nat))
            for f in [1,1.3,1.6,2,3,4,6,8,12]:
                m=mesh_for(int(k0*f))
                if m[0]>=nx and m[1]>=ny and m[2]>=nz:
                    found=int(k0*f); break
        return {"grid_density": float(found if found is not None else kppa)}
    return {mode: float(val)}

# ---------- build structure ----------
s=spec["structure"]
pot=spec.get("potcar_functional") or "PBE_64"
incar=spec.get("incar") or spec.get("incar_overrides") or {}
k_conf=spec.get("kpoints")

from pymatgen.core import Structure
if s["type"]=="path":
    structure=Structure.from_file(s["path"])
elif s["type"]=="builder":
    from pymatgen.core import Lattice
    import numpy as np
    lat=s.get("lattice",{}) or {}
    a=float(lat.get("a",3.84)); b=float(lat.get("b",3.84)); c=float(lat.get("c",3.84))
    alpha=float(lat.get("alpha",120.0)); beta=float(lat.get("beta",90.0)); gamma=float(lat.get("gamma",60.0))
    lattice=Lattice.from_parameters(a,b,c,alpha,beta,gamma)
    coords=s["coords"]
    if s.get("coord_kind","frac")=="cart":
        Minv=np.linalg.inv(lattice.matrix.T)
        coords=[list(Minv.dot(np.array(v,float))) for v in coords]
    structure=Structure(lattice, s["species"], coords)
elif s["type"]=="mp":
    mid=s["query"]; conv=bool(s.get("conventional", True)); symm=bool(s.get("symmetrize", False))
    try:
        from mp_api.client import MPRester
    except Exception as e:
        log("mp_api missing: {}".format(e)); sys.exit(2)
    key=os.environ.get("MP_API_KEY")
    if not key:
        log("MP_API_KEY not set"); sys.exit(2)
    with MPRester(key) as mpr:
        res=mpr.materials.summary.search(material_ids=[mid])
        if not res:
            log("MP-ID not found: {}".format(mid)); sys.exit(2)
        structure=res[0].structure
        if conv:
            try:
                from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
                structure=SpacegroupAnalyzer(structure, symprec=1e-3).get_conventional_standard_structure(international_monoclinic=True)
            except Exception as e:
                log("conventional transform failed: {}".format(e))
        if symm:
            try:
                from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
                structure=SpacegroupAnalyzer(structure, symprec=1e-3).get_refined_structure()
            except Exception as e:
                log("symmetrize failed: {}".format(e))
else:
    log("Unsupported structure spec: {}".format(s)); sys.exit(3)

# ---------- helpers used by steps ----------
def _samefile(a,b):
    try:
        return os.path.exists(a) and os.path.exists(b) and os.path.samefile(a,b)
    except Exception:
        return os.path.abspath(a)==os.path.abspath(b)

def _copy_from_prev(prev_dir, outdir, names=("WAVECAR","CHGCAR")):
    try:
        if not prev_dir: return
        _decompress_if_needed(prev_dir, names=names)
        for nm in names:
            src=os.path.join(prev_dir, nm)
            if os.path.exists(src):
                dst=os.path.join(outdir, nm)
                try:
                    if not _samefile(src, dst):
                        shutil.copy2(src, dst)
                        log(f"Copied {nm} from {prev_dir} -> {outdir}")
                except Exception as e:
                    log(f"Could not copy {nm} from prev_dir: {e}")
            else:
                gz=os.path.join(prev_dir, nm+".gz")
                if os.path.exists(gz):
                    shutil.copy2(gz, os.path.join(outdir, nm+".gz"))
                    log(f"Copied {nm}.gz from {prev_dir} -> {outdir}")
    except Exception as e:
        log(f"_copy_from_prev error: {e}")

def _decompress_if_needed(src_dir, names=("WAVECAR","CHGCAR")):
    for nm in names:
        p=os.path.join(src_dir, nm)
        if os.path.exists(p): continue
        gz=p+".gz"
        if os.path.exists(gz):
            try:
                with gzip.open(gz,"rb") as fin, open(p,"wb") as fout:
                    shutil.copyfileobj(fin, fout)
                log("Decompressed {} -> {}".format(os.path.basename(gz), nm))
            except Exception as e:
                log("Could not gunzip {}: {}".format(gz, e))

def _write_input(vset, outdir):
    os.makedirs(outdir, exist_ok=True)
    vset.write_input(outdir)

def _merge_user_incar(base_mode):
    safe_keys={"ENCUT","EDIFF","ISPIN","LORBIT","LREAL","ISMEAR","SIGMA","MAGMOM","EDIFFG","ADDGRID"}
    u={k:v for k,v in dict(incar).items() if k in safe_keys}
    for bad in ("NCORE","KPAR","NPAR"): u.pop(bad, None)
    return u

# ---------- atomate2 preferred for base steps ----------
use_a2=True

def _primitive_for_path(struct):
    try:
        from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
        return SpacegroupAnalyzer(struct, symprec=1e-2, angle_tolerance=5).get_primitive_standard_structure()
    except Exception:
        return struct

def _write_line_mode_kpoints(struct, out_path="KPOINTS", divisions=40):
    try:
        from pymatgen.symmetry.bandstructure import HighSymmKpath
    except Exception:
        try:
            from pymatgen.symmetry.kpath import HighSymmKpath
        except Exception:
            return False
    from pymatgen.io.vasp.inputs import Kpoints
    s_prim = _primitive_for_path(struct)
    kpath = HighSymmKpath(s_prim, symprec=1e-2, angle_tolerance=5)
    kp = Kpoints.automatic_linemode(divisions=int(divisions), ibz=kpath)
    kp.write_file(out_path)
    return True

def _custodian_handlers(for_band=False):
    from custodian.vasp.handlers import (
        VaspErrorHandler, MeshSymmetryErrorHandler, PositiveEnergyErrorHandler, FrozenJobErrorHandler,
        NonConvergingErrorHandler, UnconvergedErrorHandler
    )
    base = [VaspErrorHandler(), PositiveEnergyErrorHandler(), FrozenJobErrorHandler()]
    if for_band:
        base.append(MeshSymmetryErrorHandler())
    else:
        base.extend([MeshSymmetryErrorHandler(), NonConvergingErrorHandler(), UnconvergedErrorHandler()])
    return base

# ================== do_relax/static ==================
def do_relax(struct, name, isif=None):
    from atomate2.vasp.sets.core import RelaxSetGenerator
    from atomate2.vasp.jobs.core import RelaxMaker
    from jobflow import Flow
    from jobflow.managers.local import run_locally
    u=dict(incar)
    u.setdefault("ENCUT", ENCUT_RELAX_DEFAULT)   # 580 eV
    u.setdefault("ISPIN",2)
    u.setdefault("EDIFF",1e-6)
    u.setdefault("ADDGRID", True)
    u.setdefault("EDIFFG", -0.01)
    if isif is not None: u["ISIF"]=isif
    gen=RelaxSetGenerator(
        user_potcar_functional=pot,
        user_kpoints_settings=ksettings(struct, k_conf),
        user_incar_settings=incar_relax(u, user=incar)  # ensures HSE relax gets PRECFOCK=Fast only here
    )
    mk=RelaxMaker(input_set_generator=gen, name=("relax_ions" if isif==2 else "relax"))
    os.makedirs(name, exist_ok=True); old=os.getcwd(); os.chdir(name)
    try:
        nm = name.replace("00_","").replace("01_","").replace("02_","").strip("./")
        flow = Flow([mk.make(struct)], name=(__LABEL__ if name in (".","") else __LABEL__+"_"+nm))
        try:
            run_locally(flow, ensure_success=True, create_folders=False)
        except TypeError:
            run_locally(flow, ensure_success=True)
        c=latest(["**/INCAR","**/OUTCAR","**/vasprun.xml"], ".")
        if c:
            src_dir=os.path.dirname(c)
            if not _samefile(src_dir, "."):
                copy_keys(src_dir, ".")
    finally:
        os.chdir(old)
    try:
        from pymatgen.io.vasp.outputs import Vasprun
        v=latest([f"{name}/**/vasprun.xml", f"{name}/**/vasprun.xml.gz"])
        if v:
            return Vasprun(v, parse_potcar_file=False).final_structure
    except Exception as e:
        log("parse vasprun {}: {}".format(name, e))
    c=latest([f"{name}/**/CONTCAR", f"{name}/**/CONTCAR.gz"])
    if c:
        try:
            return Structure.from_file(c)
        except Exception as e:
            log("read CONTCAR {}: {}".format(name, e))
    return None

def do_static(struct, outdir, hse=False, prep_for_gw=False, intent="final"):
    """
    intent:
      - 'prep'  : lean step to seed next job (WAVECAR+CHGCAR, heavy maps off).
      - 'final' : report-quality outputs (tight, Accurate, richer DOS).
    """
    from atomate2.vasp.sets.core import StaticSetGenerator
    from atomate2.vasp.jobs.core import StaticMaker
    from jobflow import Flow
    from jobflow.managers.local import run_locally

    u = dict(incar)

    if hse:
        for k, v in {"LHFCALC": True, "AEXX": 0.25, "HFSCREEN": 0.2, "ALGO": "Damped"}.items():
            u.setdefault(k, v)
        for bad in ("NCORE", "KPAR", "NPAR"):
            u.pop(bad, None)

    if intent == "prep":
        u.setdefault("LWAVE", True)
        u.setdefault("LCHARG", True)
        for k in ("LVTOT", "LELF", "LVHAR", "LAECHG"):
            u.setdefault(k, False)
        try:
            encut_now = int(float(u.get("ENCUT", 0)))
        except Exception:
            encut_now = 0
        u["ENCUT"] = max(encut_now, ENCUT_STATIC_PREP_DEFAULT)  # 520
    else:  # final
        u.setdefault("LWAVE", False)
        u.setdefault("LCHARG", True)
        u.setdefault("ISMEAR", -5)
        u.setdefault("SIGMA", 0.05)
        u.setdefault("NEDOS", 4001)
        u.setdefault("LORBIT", 11)
        u.setdefault("LREAL", False)
        u.setdefault("PREC", "Accurate")
        u.setdefault("ADDGRID", True)
        try:
            encut_now = int(float(u.get("ENCUT", 0)))
        except Exception:
            encut_now = 0
        u["ENCUT"] = max(encut_now, ENCUT_STATIC_FINAL_DEFAULT)  # 620

    if prep_for_gw:
        u["LWAVE"] = True
        u["LCHARG"] = True

    gen = StaticSetGenerator(
        user_potcar_functional=pot,
        user_kpoints_settings=ksettings(struct, k_conf),
        user_incar_settings=incar_static(u, allow_ncore=not (hse or prep_for_gw)),
    )
    mk = StaticMaker(input_set_generator=gen, name=("hse_static" if hse else "static"))

    os.makedirs(outdir, exist_ok=True)
    old = os.getcwd(); os.chdir(outdir)
    try:
        flow = Flow([mk.make(struct)], name=__LABEL__ + ("_hse_static" if hse else "_static"))
        try:
            run_locally(flow, ensure_success=True, create_folders=False)
        except TypeError:
            run_locally(flow, ensure_success=True)
        c = latest(["**/INCAR", "**/OUTCAR", "**/vasprun.xml"], ".")
        if c:
            src_dir = os.path.dirname(c)
            if not _samefile(src_dir, "."):
                copy_keys(src_dir, ".")
    finally:
        os.chdir(old)

    try:
        if outdir not in (".", ""):
            if os.path.islink("latest_calc") or os.path.exists("latest_calc"):
                try: os.unlink("latest_calc")
                except Exception: pass
            os.symlink(outdir, "latest_calc", target_is_directory=True)
    except Exception:
        pass

    for f in ("INCAR","KPOINTS","POSCAR","CONTCAR","OUTCAR","vasprun.xml","CHGCAR","WAVECAR"):
        for c in (os.path.join(outdir, f + ".gz"), os.path.join(outdir, f)):
            if os.path.exists(c):
                dst = os.path.join(outdir, (f if not c.endswith(".gz") else f + ".gz"))
                try:
                    if not _samefile(c, dst):
                        shutil.copy2(c, dst)
                except Exception:
                    pass
                break

    if intent == "prep":
        chg = os.path.join(outdir, "CHGCAR")
        try:
            if os.path.exists(chg) and os.path.getsize(chg) < 1024:
                os.remove(chg)
                log(f"[static:{intent}] Removed empty CHGCAR in {outdir}")
        except Exception as e:
            log(f"[static:{intent}] Could not clean CHGCAR: {e}")

# ================== do_bands() ==================
def do_bands(struct, outdir, prev_dir=None, hse=False):
    from pymatgen.io.vasp.sets import MPNonSCFSet
    from custodian.custodian import Custodian
    from custodian.vasp.jobs import VaspJob
    os.makedirs(outdir, exist_ok=True)
    if prev_dir:
        _copy_from_prev(prev_dir, outdir, names=("WAVECAR","CHGCAR"))
    _decompress_if_needed(outdir, names=("WAVECAR","CHGCAR"))

    for nm in ("WAVECAR","CHGCAR"):
        p = os.path.join(outdir, nm)
        if os.path.exists(p) and os.path.getsize(p) < 1024:
            try:
                os.remove(p)
                log(f"Removed empty {nm} in {outdir}")
            except Exception as e:
                log(f"Could not remove empty {nm}: {e}")

    wav = os.path.join(outdir, "WAVECAR"); has_wav = os.path.exists(wav) and os.path.getsize(wav) > 1024
    chg = os.path.join(outdir, "CHGCAR"); has_chg = os.path.exists(chg) and os.path.getsize(chg) > 1024

    if not hse:
        incar_overrides = {"NSW": 0, "LCHARG": False, "LWAVE": False, "ICHARG": 0}
        if has_wav: incar_overrides["ISTART"] = 1
    else:
        if not has_wav:
            raise RuntimeError("HSE bands requires WAVECAR from prior HSE static in prev_dir.")
        incar_overrides = {
            "NSW":0,"LCHARG":False,"LWAVE":False,
            "ISTART":1,"NELM":1,
            "LHFCALC":True,"AEXX":0.25,"HFSCREEN":0.2,
            "ALGO":"Damped","LREAL":False,
        }
        incar_overrides["ICHARG"] = 11 if has_chg else 0
        for bad in ("NCORE","KPAR","NPAR"): incar_overrides.pop(bad, None)
    incar_overrides["ISYM"]=0; incar_overrides["SYMPREC"]=1e-8

    old = os.getcwd(); os.chdir(outdir)
    try:
        vset = MPNonSCFSet(struct, mode="Line", user_incar_settings=incar_overrides, user_potcar_functional=pot)
        vset.write_input(".")
        wrote = _write_line_mode_kpoints(struct, out_path="KPOINTS", divisions=40)
        if not wrote:
            log("Could not import HighSymmKpath; using KPOINTS from MPNonSCFSet(mode='Line').")
        cmd=_build_vasp_cmd()
        handlers=_custodian_handlers(for_band=True)
        jobs=[VaspJob(cmd, auto_npar=False, output_file="vasp.out", stderr_file="std_err.txt")]
        Custodian(handlers, jobs, max_errors=10, checkpoint=True).run()
        copy_keys(".", ".")
    finally:
        os.chdir(old)
    print("JOBFLOW_LOCAL_DONE")

# ================== do_bands_gw_true() ==================
def do_bands_gw_true(struct, outdir, prev_dir):
    from pymatgen.io.vasp.sets import MVLGWSet
    from custodian.custodian import Custodian
    from custodian.vasp.jobs import VaspJob
    os.makedirs(outdir, exist_ok=True)

    user_bands = {
        "NSW": 0, "ISMEAR": -5, "SIGMA": 0.01,
        "ISTART": 1, "ICHARG": 11, "ALGO": "Normal", "NELM": 1,
        "ISYM": 0, "SYMPREC": 1e-8, "LREAL": False, "LWAVE": False, "LCHARG": False,
    }
    vset_bs = MVLGWSet.from_prev_calc(prev_dir, mode="STATIC",
                                      user_incar_settings=user_bands,
                                      user_kpoints_settings=None,
                                      user_potcar_functional=pot)
    _write_input(vset_bs, outdir)

    _copy_from_prev(prev_dir, outdir, names=("WAVECAR","CHGCAR"))
    _decompress_if_needed(outdir, names=("WAVECAR","CHGCAR"))
    wrote = _write_line_mode_kpoints(struct, out_path=os.path.join(outdir, "KPOINTS"), divisions=40)

    try:
        with open(os.path.join(outdir, "README_gw_bands.txt"), "w") as f:
            f.write(
                "This step is a non-self-consistent DFT band structure along a high-symmetry path.\n"
                "VASP GW requires a uniform k-mesh and refuses line-mode KPOINTS.\n"
                "Use GW data from 01_gw_mvl (WFULL*, vasprun.xml) to scissor or interpolate QP energies.\n"
                f"KPOINTS line-mode written: {'yes' if wrote else 'no (used default)'}\n"
            )
    except Exception:
        pass

    old = os.getcwd(); os.chdir(outdir)
    try:
        cmd=_build_vasp_cmd()
        handlers=_custodian_handlers(for_band=True)
        jobs=[VaspJob(cmd, auto_npar=False, output_file="vasp.out", stderr_file="std_err.txt")]
        Custodian(handlers, jobs, max_errors=10, checkpoint=True).run()
        copy_keys(".", ".")
    finally:
        os.chdir(old)
    print("JOBFLOW_LOCAL_DONE (DFT line-mode using GW charge density)")

# ---------- dispatcher ----------
try:
    if wf=="static":
        do_static(structure, ".", hse=_is_hse_incar(incar), intent="final")
        print("JOBFLOW_LOCAL_DONE")

    elif wf in ("relax","relax_ions"):
        s = do_relax(structure, ".", isif=(2 if wf=="relax_ions" else None))
        print("JOBFLOW_LOCAL_DONE")

    elif wf in ("relax_static","relax_ions_static"):
        s = do_relax(structure, "00_relax" if wf=="relax_static" else "00_relax_ions", isif=(2 if wf=="relax_ions_static" else None))
        if s is None:
            log("Could not obtain relaxed structure; aborting static."); sys.exit(4)
        do_static(s, "01_static", hse=False, intent="final")
        print("JOBFLOW_LOCAL_DONE")

    elif wf=="relax_static_bands":
        s = do_relax(structure, "00_relax")
        if s is None:
            log("Could not obtain relaxed structure; aborting."); sys.exit(4)
        do_static(s, "01_hse_static", hse=True, intent="prep")
        try:
            wv = os.path.join("01_hse_static", "WAVECAR")
            if not (os.path.exists(wv) and os.path.getsize(wv) > 1024):
                log("01_hse_static didn't produce a valid WAVECAR; check INCAR/LWAVE and vasp.out")
                sys.exit(5)
        except Exception:
            pass
        do_bands(s, "02_bands_hse", prev_dir="01_hse_static", hse=True)
        print("JOBFLOW_LOCAL_DONE")

    elif wf=="gw_static":
        from pymatgen.io.vasp.sets import MVLGWSet
        from custodian.custodian import Custodian
        from custodian.vasp.jobs import VaspJob
        from custodian.vasp.handlers import VaspErrorHandler, MeshSymmetryErrorHandler, NonConvergingErrorHandler, UnconvergedErrorHandler, PositiveEnergyErrorHandler, FrozenJobErrorHandler

        def _run_with_custodian():
            cmd=_build_vasp_cmd()
            handlers=[VaspErrorHandler(), MeshSymmetryErrorHandler(), NonConvergingErrorHandler(), UnconvergedErrorHandler(), PositiveEnergyErrorHandler(), FrozenJobErrorHandler()]
            Custodian(handlers, [VaspJob(cmd, auto_npar=False, output_file="vasp.out", stderr_file="std_err.txt")], max_errors=10, checkpoint=True).run()

        def _run_step(dirpath):
            old=os.getcwd(); os.chdir(dirpath)
            try: _run_with_custodian()
            finally: os.chdir(old)

        ks = ksettings(structure, k_conf)

        prep_dir="00_gw_mvl"; gw_dir="01_gw_mvl"; os.makedirs(gw_dir, exist_ok=True)
        u_prep = {**_merge_user_incar("STATIC"),
                  "LWAVE": True, "LCHARG": True,
                  "LVTOT": False, "LELF": False, "LVHAR": False, "LAECHG": False}
        try:
            _enc = int(float(u_prep.get("ENCUT", 0)))
        except Exception:
            _enc = 0
        u_prep["ENCUT"] = max(_enc, ENCUT_STATIC_PREP_DEFAULT)

        log("Starting MVL GW: static prep -> " + prep_dir)
        vset_static = MVLGWSet(structure, mode="STATIC", user_potcar_functional=pot,
                               user_incar_settings=u_prep, user_kpoints_settings=ks)
        _write_input(vset_static, prep_dir)
        _run_step(prep_dir)
        _decompress_if_needed(prep_dir, names=("WAVECAR","CHGCAR"))

        log("Running MVL GW -> " + gw_dir)
        user_gw = _merge_user_incar("GW"); user_gw["LREAL"] = False
        vset_gw = MVLGWSet.from_prev_calc(prep_dir, mode="GW",
                                          user_incar_settings=user_gw,
                                          user_kpoints_settings=ks,
                                          user_potcar_functional=pot)
        _write_input(vset_gw, gw_dir)
        _copy_from_prev(prep_dir, gw_dir, names=("WAVECAR","CHGCAR"))
        _run_step(gw_dir)
        print("JOBFLOW_LOCAL_DONE (MVL two-stage GW chain completed)")

    elif wf=="gw_static_bands_true":
        from pymatgen.io.vasp.sets import MVLGWSet
        from custodian.custodian import Custodian
        from custodian.vasp.jobs import VaspJob
        from custodian.vasp.handlers import VaspErrorHandler, MeshSymmetryErrorHandler, NonConvergingErrorHandler, UnconvergedErrorHandler, PositiveEnergyErrorHandler, FrozenJobErrorHandler

        def _run_with_custodian():
            cmd=_build_vasp_cmd()
            handlers=[VaspErrorHandler(), MeshSymmetryErrorHandler(), NonConvergingErrorHandler(),
                      UnconvergedErrorHandler(), PositiveEnergyErrorHandler(), FrozenJobErrorHandler()]
            Custodian(handlers, [VaspJob(cmd, auto_npar=False, output_file="vasp.out", stderr_file="std_err.txt")], max_errors=10, checkpoint=True).run()

        def _run_step(dirpath):
            old=os.getcwd(); os.chdir(dirpath)
            try: _run_with_custodian()
            finally: os.chdir(old)

        ks = ksettings(structure, k_conf)

        prep_dir = "00_gw_mvl"; gw_dir = "01_gw_mvl"; out_dir = "02_bands_gw_true"
        os.makedirs(gw_dir, exist_ok=True)

        u_prep = {**_merge_user_incar("STATIC"),
                  "LWAVE": True, "LCHARG": True,
                  "LVTOT": False, "LELF": False, "LVHAR": False, "LAECHG": False}
        try:
            _enc = int(float(u_prep.get("ENCUT", 0)))
        except Exception:
            _enc = 0
        u_prep["ENCUT"] = max(_enc, ENCUT_STATIC_PREP_DEFAULT)

        log("Starting MVL GW: static prep -> " + prep_dir)
        vset_static = MVLGWSet(structure, mode="STATIC", user_potcar_functional=pot,
                               user_incar_settings=u_prep, user_kpoints_settings=ks)
        _write_input(vset_static, prep_dir)
        _run_step(prep_dir)
        _decompress_if_needed(prep_dir, names=("WAVECAR","CHGCAR"))

        log("Running MVL GW -> " + gw_dir)
        user_gw = _merge_user_incar("GW"); user_gw["LREAL"] = False
        vset_gw = MVLGWSet.from_prev_calc(prep_dir, mode="GW",
                                          user_incar_settings=user_gw,
                                          user_kpoints_settings=ks,
                                          user_potcar_functional=pot)
        _write_input(vset_gw, gw_dir)
        _copy_from_prev(prep_dir, gw_dir, names=("WAVECAR","CHGCAR"))
        _run_step(gw_dir)

        log("Starting band path (DFT, non-SCF) -> " + out_dir)
        do_bands_gw_true(structure, out_dir, prev_dir=gw_dir)
        print("JOBFLOW_LOCAL_DONE")

    elif wf in ("relax2_static","relax2_hse_static"):
        s1 = do_relax(structure, "00_relax1")
        if s1 is None:
            log("Could not obtain structure after Relax1; aborting."); sys.exit(4)
        s2 = do_relax(s1, "01_relax2")
        if s2 is None:
            log("Could not obtain structure after Relax2; aborting."); sys.exit(4)
        do_static(s2, ("02_hse_static" if wf=="relax2_hse_static" else "02_static"),
                  hse=(wf=="relax2_hse_static"), intent="final")
        print("JOBFLOW_LOCAL_DONE")

    else:
        raise ValueError("Unknown workflow: "+str(wf))

except Exception as e:
    # Disable global fallback for complex chains
    if wf in ("gw_static_bands_true", "relax_static_bands", "gw_static"):
        log(f"Fatal error in workflow '{wf}'; fallback disabled. Error: {e}")
        traceback.print_exc(file=sys.stderr)
        sys.exit(9)

    log("Falling back to MP* sets + Custodian due to: {}".format(e))
    try:
        from custodian.custodian import Custodian
        from custodian.vasp.jobs import VaspJob
        from custodian.vasp.handlers import VaspErrorHandler, MeshSymmetryErrorHandler, NonConvergingErrorHandler, UnconvergedErrorHandler, PositiveEnergyErrorHandler, FrozenJobErrorHandler
        from pymatgen.io.vasp.sets import MPStaticSet, MPRelaxSet
        from pymatgen.core import Structure as _S
        def run_vasp_here():
            cmd=_build_vasp_cmd()
            handlers=[VaspErrorHandler(), MeshSymmetryErrorHandler(), NonConvergingErrorHandler(),
                      UnconvergedErrorHandler(), PositiveEnergyErrorHandler(), FrozenJobErrorHandler()]
            Custodian(handlers, [VaspJob(cmd, auto_npar=False, output_file="vasp.out", stderr_file="std_err.txt")], max_errors=10, checkpoint=True).run()
        def read_final_structure(d="."):
            try:
                from pymatgen.io.vasp.outputs import Vasprun
                v=latest([f"{d}/**/vasprun.xml", f"{d}/**/vasprun.xml.gz"])
                if v: return Vasprun(v, parse_potcar_file=False).final_structure
            except Exception: pass
            c=latest([f"{d}/**/CONTCAR", f"{d}/**/CONTCAR.gz"])
            if c:
                try:
                    return _S.from_file(c)
                except Exception: pass
            return None
        ks = ksettings(structure, spec.get("kpoints"))
        if wf=="static":
            outdir="00_static"
            _incar_f = dict(incar)
            try:
                _enc = int(float(_incar_f.get("ENCUT", 0)))
            except Exception:
                _enc = 0
            _incar_f["ENCUT"] = max(_enc, ENCUT_STATIC_FINAL_DEFAULT)  # 620
            _incar_f.setdefault("ADDGRID", True)
            vset = MPStaticSet(structure, user_potcar_functional=pot, user_incar_settings=incar_static(_incar_f), user_kpoints_settings=ks)
            _write_input(vset, outdir)
            old=os.getcwd(); os.chdir(outdir)
            try: run_vasp_here()
            finally: os.chdir(old)
            copy_keys(outdir, ".")
            print("JOBFLOW_LOCAL_DONE (fallback static)")
        elif wf in ("relax","relax_ions"):
            outdir="00_relax" if wf=="relax" else "00_relax_ions"
            u=dict(incar_relax(incar, user=incar))
            try:
                _enc = int(float(u.get("ENCUT", 0)))
            except Exception:
                _enc = 0
            if _enc < ENCUT_RELAX_DEFAULT:
                u["ENCUT"] = ENCUT_RELAX_DEFAULT  # 580
            if wf=="relax_ions": u["ISIF"]=2
            if "LCHARG" not in incar: u["LCHARG"] = False
            if "LWAVE"  not in incar: u["LWAVE"]  = False
            u.setdefault("ADDGRID", True)
            u.setdefault("EDIFFG", -0.01)
            vset = MPRelaxSet(structure, user_potcar_functional=pot, user_incar_settings=u, user_kpoints_settings=ks)
            _write_input(vset, outdir)
            old=os.getcwd(); os.chdir(outdir)
            try: run_vasp_here()
            finally: os.chdir(old)
            copy_keys(outdir, ".")
            print("JOBFLOW_LOCAL_DONE (fallback relax)")
        elif wf in ("relax_static","relax_ions_static"):
            rdir="00_relax" if wf=="relax_static" else "00_relax_ions"
            u=dict(incar_relax(incar, user=incar))
            try:
                _enc = int(float(u.get("ENCUT", 0)))
            except Exception:
                _enc = 0
            if _enc < ENCUT_RELAX_DEFAULT:
                u["ENCUT"] = ENCUT_RELAX_DEFAULT  # 580
            if wf=="relax_ions_static": u["ISIF"]=2
            if "LCHARG" not in incar: u["LCHARG"] = False
            if "LWAVE"  not in incar: u["LWAVE"]  = False
            u.setdefault("ADDGRID", True)
            u.setdefault("EDIFFG", -0.01)
            vset = MPRelaxSet(structure, user_potcar_functional=pot, user_incar_settings=u, user_kpoints_settings=ks)
            _write_input(vset, rdir)
            old=os.getcwd(); os.chdir(rdir)
            try: run_vasp_here()
            finally: os.chdir(old)
            s_rel = read_final_structure(rdir)
            if s_rel is None: log("Fallback: could not read relaxed structure; stopping."); sys.exit(7)
            sdir="01_static"
            _incar_f = dict(incar)
            try:
                _enc = int(float(_incar_f.get("ENCUT", 0)))
            except Exception:
                _enc = 0
            _incar_f["ENCUT"] = max(_enc, ENCUT_STATIC_FINAL_DEFAULT)  # 620
            _incar_f.setdefault("ADDGRID", True)
            vset2 = MPStaticSet(s_rel, user_potcar_functional=pot, user_incar_settings=incar_static(_incar_f), user_kpoints_settings=ks)
            _write_input(vset2, sdir)
            old=os.getcwd(); os.chdir(sdir)
            try: run_vasp_here()
            finally: os.chdir(old)
            copy_keys(rdir, sdir); copy_keys(sdir, ".")
            print("JOBFLOW_LOCAL_DONE (fallback relax->static)")
        else:
            log("Fallback only implements static/relax(+static); requested wf='{}'".format(wf))
            sys.exit(8)
    except Exception as ee:
        log("Fallback chain error: {}".format(ee))
        traceback.print_exc(file=sys.stderr)
        sys.exit(1)
'''.replace("__SPEC__", repr(spec_json)).replace("__LABEL__", repr(label))

    # Widget-provided MP_API_KEY only
    key_from_widget = (mp_api_key or "").strip()
    mp_export = f'export MP_API_KEY={shlex.quote(key_from_widget)}' if key_from_widget else 'echo "[sbatch] MP_API_KEY not provided for this run."'

    # ---- Resolve SLURM partition/account with sensible fallbacks ----
    part = (getattr(cfg, "partition", None)
            or os.environ.get("SLURM_PARTITION")
            or os.environ.get("SLURM_DEFAULT_PARTITION")
            or "power-leeburton")
    acct = (getattr(cfg, "account", None)
            or os.environ.get("SLURM_ACCOUNT")
            or "power-leeburton-users")

    # ---- Export runtime env explicitly ----
    _default_launcher = "srun --mpi=pmi2 -n $SLURM_NTASKS vasp_std"
    vasp_cmd_literal = (
        os.environ.get("VASP_CMD")
        or getattr(cfg, "VASP_CMD", "").strip()
        or _default_launcher
    )
    psp_dir_literal  = os.environ.get("PMG_VASP_PSP_DIR", getattr(cfg, "potcars_dir", ""))
    jf_cfg_literal   = os.environ.get("JOBFLOW_CONFIG_FILE", getattr(cfg, "jobflow_config_file", ""))

    job_body = f"""
set -e -o pipefail
mkdir -p "{run_dir}"
cd "{run_dir}"
{mp_export}
# Disable gzip both in custodian and atomate2
export CUSTODIAN_NO_GZIP=1
export ATOMATE2_VASP_ZIP_FILES=False
# Critical env for VASP + inputs
export VASP_CMD='{vasp_cmd_literal}'
{f'export PMG_VASP_PSP_DIR={shlex.quote(psp_dir_literal)}' if psp_dir_literal else 'echo "[sbatch] PMG_VASP_PSP_DIR not set"'}
{f'export JOBFLOW_CONFIG_FILE={shlex.quote(jf_cfg_literal)}' if jf_cfg_literal else 'true'}
echo "[sbatch] Using partition={part} account={acct}"
echo "[sbatch] VASP_CMD=$VASP_CMD"
echo "[sbatch] SLURM_NTASKS=${{SLURM_NTASKS:-<unset>}}"
which srun 2>/dev/null || true; srun --version 2>/dev/null | head -n1 || true
which {_env_bin(cfg)}/python || true
python --version || true
cat > run_job.py <<'PY'
{job_python}
PY
{_env_bin(cfg)}/python -u run_job.py 1>"{log_out}" 2>"{log_err}"
echo "Done. Logs:"; echo "{log_out}"; echo "{log_err}"
""".lstrip("\n")

    base_sbatch = make_sbatch(cfg, job_body)
    header = (
        f"#SBATCH --job-name={run_name}\n"
        f"#SBATCH --output={posixpath.join(cfg.logs_dir, run_name + '.slurm.out')}\n"
        f"#SBATCH --error={posixpath.join(cfg.logs_dir, run_name + '.slurm.err')}\n"
    )
    sbatch_script = _inject_header(base_sbatch, header)
    remote_script = posixpath.join(cfg.flows_dir, f"{run_name}.sbatch.sh")

    mem_gb = _coerce_int(mem_gb, 120); ntasks = _coerce_int(ntasks, 24); nodes = _coerce_int(nodes, 1); wall = wall or "72:00:00"
    link_map = {"PBE_64":["POT_GGA_PAW_PBE_64","POT_PAW_PBE_64"], "PBE_54":["POT_GGA_PAW_PBE_54"], "PBE_52":["POT_GGA_PAW_PBE_52","POT_GGA_PAW_PBE"], "LDA":["POT_LDA_PAW"]}
    pot_links = " ".join(posixpath.join(cfg.potcars_dir, L) for L in link_map.get(pot_func, []))
    remote_script_q = shlex.quote(remote_script)
    sbatch_line = (
        f'sbatch -p {shlex.quote(str(part))} -A {shlex.quote(str(acct))} '
        f'-N {nodes} -n {ntasks} --mem={mem_gb}G -t {wall} --parsable {remote_script_q}'
    )
    cmd = f"""set -e -o pipefail
mkdir -p "{cfg.flows_dir}" "{cfg.logs_dir}" "{cfg.potcars_dir}" "{pot_target}"
for L in {pot_links}; do ln -sfn "{pot_target}" "$L"; done
cat > "{remote_script}" <<'SBATCH'
{sbatch_script}
SBATCH
{"echo DRY RUN; exit 0" if dry_run else ""}
echo "Submitting with: {sbatch_line}"
out=$({sbatch_line} 2>&1); rc=$?; echo "SBATCH_RAW_OUT=$out"; exit $rc
"""
    if dry_run:
        return {"sbatch_preview": sbatch_script, "run_dir": run_dir, "log_out": log_out, "log_err": log_err}

    rc, out, err = r.run(cmd, check=False, modules=False, export_env=False)
    if rc != 0:
        raise RuntimeError((err or out).strip() or "Submit step failed")
    m = (re.search(r"(?m)^JOBID=(\d+)\b", out) or
         re.search(r"(?m)^SBATCH_RAW_OUT=(\d+)\b", out) or
         re.search(r"(?m)^(\d+)\b", out) or
         re.search(r"Submitted batch job\s+(\d+)", "")
        )
    if not m:
        m = (re.search(r"Submitted batch job\s+(\d+)", out) or re.search(r"\b(\d+)\b", out))
    if not m:
        raise RuntimeError("Could not parse job id from sbatch output above.")
    return {"jobid": m.group(1), "run_dir": run_dir, "log_out": log_out, "log_err": log_err}

# ========= UI =========
wHTML = w.HTML
cfg_defaults = dict(nodes=1, ntasks=24, mem_gb=120, time="72:00:00", functional="PBE_64")

input_method = w.Dropdown(
    options=[("I have a POSCAR already","path"),("Build POSCAR with PyMatGen","builder"),("Retrieve from the Materials Project","mp")],
    value="path", description="Input method"
)
path_label = w.Label("Enter the full path to the POSCAR on the cluster:"); structure_path = w.Text(value="/path/on/remote/POSCAR", description="Path")

# ---- builder defaults ----
a_len=b_len=c_len=w.FloatText(value=3.84,description="a/b/c [Angstrom]")
alpha=w.FloatText(value=120.0,description="alpha [deg]"); beta=w.FloatText(value=90.0,description="beta [deg]"); gamma=w.FloatText(value=60.0,description="gamma [deg]")
species_txt=w.Text(value="Si,Si",description="species")
coord_kind=w.Dropdown(options=[("fractional","frac"),("cartesian","cart")],value="frac",description="coord type")
coords_txt=w.Textarea(value="0 0 0\n0.75 0.5 0.75",description="coords",layout={"height":"110px"})
builder_box=w.VBox([w.HBox([a_len,b_len,c_len]), w.HBox([alpha,beta,gamma]), w.HBox([species_txt,coord_kind]), coords_txt])

# --- MP widgets ---
mp_id_tb=w.Text(value="mp-149",description="MP-ID")
mp_conv=w.Checkbox(value=True,description="Conventional")
mp_symm=w.Checkbox(value=False,description="Symmetrize")
mp_api_key_tb=w.Password(value=os.environ.get("MP_API_KEY",""), description="MP_API_KEY", placeholder="required for MP retrieval")
mp_box=w.VBox([
    mp_id_tb,
    w.HBox([mp_conv, mp_symm]),
    mp_api_key_tb,
    w.HTML("<i>Only the value in this field is used for submission. No environment fallback.</i>")
])

# === WORKFLOW SELECTOR ===
wf = w.Dropdown(
    options=[
        ("Static (final)", "static"),

        # Single relax variants
        ("Relax (ions only)", "relax_ions"),
        ("Relax (ions+cell)", "relax"),

        # HSE single-point / relax variants
        ("HSE06 Static (final)", "hse_static"),
        ("HSE06 Relax (ions only)", "hse_relax_ions"),
        ("HSE06 Relax (ions+cell)", "hse_relax"),

        # Relax -> Static chains
        ("Relax (ions only) → Static (final)", "relax_ions_static"),
        ("Relax (ions+cell) → Static (final)", "relax_static"),

        # Relax -> Relax -> Static/HSE06 chains
        ("Relax (ions+cell) → Relax (ions+cell) → Static (final)", "relax2_static"),
        ("Relax (ions+cell) → Relax (ions+cell) → HSE06 Static (final)", "relax2_hse_static"),

        # Bands/GW chains
        ("Relax (ions+cell) → HSE06 Static (prep) → HSE06 Bands (line-mode)", "relax_static_bands"),
        ("Static (prep, MVL) → GW (MVL)", "gw_static"),
        ("Static (prep, MVL) → GW (MVL) → DFT Bands via GW charge (line-mode)", "gw_static_bands_true"),
    ],
    value="static",
    description="Workflow"
)

functional = w.Dropdown(options=["PBE_64","PBE_54","PBE_52","LDA"], value=cfg_defaults["functional"], description="POTCAR")

k_mode = w.Dropdown(options=[("workflow default","workflow default"),("Mesh (nx ny nz)","mesh"),("reciprocal_density","reciprocal_density"),("grid_density","grid_density"),("length","length")], value="workflow default", description="KPOINTS")
k_mesh = w.Text(value="workflow default", description="Mesh / Value", disabled=True)

encut=w.Text(value="workflow default", description="ENCUT")
ediff=w.Text(value="workflow default", description="EDIFF")
ediffg=w.Text(value="workflow default", description="EDIFFG")
ispin=w.Dropdown(options=[("workflow default","workflow default"),("1 (non-spin)","1"),("2 (spin)","2")], value="workflow default", description="ISPIN")

nodes=w.IntText(value=cfg_defaults["nodes"], description="nodes")
ntasks=w.IntText(value=cfg_defaults["ntasks"], description="cpus")
mem_gb=w.IntText(value=cfg_defaults["mem_gb"], description="memory(GB)")
wall=w.Text(value=cfg_defaults["time"], description="time limit")
job_label=w.Text(value="vasp_run", description="Job label")

dry=w.Checkbox(value=False, description="Dry run (do not submit)")
btn=w.Button(description="Build & Submit", button_style="primary")
out_box=w.Output(); err_box=w.Output()

def _apply_wf_defaults():
    try:
        if wf.value in ("gw_static", "gw_static_bands_true"):
            if int(ntasks.value) == int(cfg_defaults["ntasks"]):
                ntasks.value = 12
            if int(mem_gb.value) == int(cfg_defaults["mem_gb"]):
                mem_gb.value = 240
        elif wf.value == "relax_static_bands":
            if int(mem_gb.value) == int(cfg_defaults["mem_gb"]):
                mem_gb.value = 160
        else:
            if int(ntasks.value) == 12:
                ntasks.value = int(cfg_defaults["ntasks"])
            if int(mem_gb.value) == 240:
                mem_gb.value = int(cfg_defaults["mem_gb"])
    except Exception:
        pass

def _toggle(*_):
    mode=input_method.value
    path_label.layout.display=None if mode=="path" else "none"
    structure_path.layout.display=None if mode=="path" else "none"
    builder_box.layout.display=None if mode=="builder" else "none"
    mp_box.layout.display=None if mode=="mp" else "none"
    if k_mode.value=="workflow default":
        k_mesh.value="workflow default"; k_mesh.disabled=True
    elif k_mode.value=="mesh":
        if k_mesh.value.strip().lower() in ("workflow default",""): k_mesh.value="2 2 2"
        k_mesh.disabled=False
    else:
        if k_mesh.value.strip().lower() in ("workflow default","2 2 2",""): k_mesh.value="100"
        k_mesh.disabled=False

for wdg in (input_method, k_mode):
    wdg.observe(lambda c: _toggle(), names=["value"])
wf.observe(lambda c: (_apply_wf_defaults(), _toggle()), names=["value"])
_toggle()

ui = w.VBox([
    wHTML("<b>Structure file (POSCAR):</b>"), input_method, path_label, structure_path,
    builder_box, mp_box,
    wHTML("<hr><b>Workflow & parameters:</b>"),
    w.HTML("<i>GW uses the MVL recipe (STATIC → GW). The <b>GW BandStructure</b> option runs a DFT line-mode bands step using the GW charge density (VASP GW forbids line-mode directly).</i>"),
    w.HBox([wf, functional]),
    w.HBox([k_mode, k_mesh]),
    w.HBox([encut, ediff, ediffg]), ispin,
    wHTML("<b>SLURM settings:</b>"), w.HBox([nodes, ntasks, mem_gb]), wall, job_label,
    w.HBox([dry]), btn,
    out_box, err_box
])

def _on_click(_):
    btn.disabled=True
    try:
        with out_box:
            print("Click received. Preparing submission...", flush=True)
        if not all(k in globals() for k in ("cfg","rmt","make_sbatch")):
            raise RuntimeError("Missing cfg/rmt/make_sbatch from Cell 1. Run Cell 1 first.")

        # ----- structure spec -----
        mode=input_method.value
        if mode=="path":
            p=structure_path.value.strip()
            if not p or p=="/path/on/remote/POSCAR":
                raise ValueError("Please provide a valid remote POSCAR path.")
            struct_spec={"type":"path","path":p}
        elif mode=="builder":
            species=[s.strip() for s in (species_txt.value or "").split(",") if s.strip()]
            if not species:
                raise ValueError("Provide at least one species, e.g., 'Si,Si'.")
            coords=[]
            for ln in (ln.strip() for ln in (coords_txt.value or "").splitlines() if ln.strip()):
                pr=ln.replace(",", " ").split()
                if len(pr)!=3:
                    raise ValueError("Each coord line must have 3 numbers")
                coords.append([float(pr[0]), float(pr[1]), float(pr[2])])
            if len(coords)!=len(species):
                raise ValueError("species and coords must have the same length")
            lattice=dict(a=float(a_len.value), b=float(b_len.value), c=float(c_len.value),
                         alpha=float(alpha.value), beta=float(beta.value), gamma=float(gamma.value))
            struct_spec={"type":"builder","kind":"structure","coord_kind":coord_kind.value,"species":species,"coords":coords,"lattice":lattice}
        else:
            mpid=mp_id_tb.value.strip()
            if not mpid:
                raise ValueError("Please enter a valid MP-ID (e.g., mp-149).")
            key_for_this_run = mp_api_key_tb.value.strip()
            if not key_for_this_run:
                raise ValueError("MP_API_KEY is required for Materials Project retrieval.")
            struct_spec={"type":"mp","by":"material_id","query":mpid,"conventional":bool(mp_conv.value),"symmetrize":bool(mp_symm.value)}

        # ----- workflow + INCAR/KPOINTS -----
        selected_wf=wf.value
        is_hse=selected_wf.startswith("hse_")
        base_wf=selected_wf.split("_",1)[1] if is_hse else selected_wf

        incar={}
        if encut.value.strip().lower()!="workflow default":
            v=int(float(encut.value)); assert v>0; incar["ENCUT"]=v
        if ediff.value.strip().lower()!="workflow default":
            v=float(ediff.value); assert v>0; incar["EDIFF"]=v
        if ediffg.value.strip().lower()!="workflow default":
            incar["EDIFFG"]=float(ediffg.value)
        if ispin.value!="workflow default":
            v=int(ispin.value); assert v in (1,2); incar["ISPIN"]=v
        if is_hse:
            for k,v in {"LHFCALC": True, "AEXX": 0.25, "HFSCREEN": 0.2, "ALGO": "Damped"}.items():
                if k not in incar:
                    incar[k]=v
            incar.pop("NCORE", None)

        if k_mode.value=="workflow default":
            kpoints=None
        elif k_mode.value=="mesh":
            parts=k_mesh.value.replace(",", " ").split()
            try:
                nx,ny,nz=(int(parts[i]) for i in range(3))
                if min(nx,ny,nz)<=0:
                    raise ValueError
            except Exception:
                raise ValueError("For mesh, provide three positive ints, e.g., '2 2 2'.")
            kpoints={"mode":"mesh","value":(nx,ny,nz)}
        else:
            try:
                val=float(k_mesh.value); assert val>0
            except Exception:
                raise ValueError(f"KPOINTS value must be a positive number for mode '{k_mode.value}'.")
            kpoints={"mode":k_mode.value,"value":val}

        if job_label.value.strip() in ("","vasp_run"):
            base = posixpath.basename(struct_spec["path"]) if mode=="path" else ("vasp" if mode=="builder" else "structure")
            job_label.value = _sanitize_label(f"{base}_{selected_wf}")
        label = _sanitize_label(job_label.value)

        nd, nt, mg = int(nodes.value), int(ntasks.value), int(mem_gb.value)
        wl = wall.value or cfg_defaults["time"]
        flow_spec={"workflow":base_wf,"potcar_functional":functional.value,"kpoints":kpoints,"incar":incar,"structure":struct_spec}

        key_for_this_run = (mp_api_key_tb.value.strip() if mode=="mp" else "")

        if dry.value:
            with out_box:
                print("Dry run — generating sbatch preview...", flush=True)
            res=fast_submit_from_spec(flow_spec, label, nt, mg, wl, nodes=nd, dry_run=True, mp_api_key=key_for_this_run)
            with out_box:
                display(Markdown("### sbatch header preview"))
                display(HTML("<pre style='white-space:pre-wrap'>"+res.get("sbatch_preview","<no preview>")+"</pre>"))
            return

        with out_box:
            print("Submitting job via sbatch...", flush=True)
        res=fast_submit_from_spec(flow_spec, label, nt, mg, wl, nodes=nd, mp_api_key=key_for_this_run)

        # === Persist job state ===
        run_name = posixpath.basename(res["run_dir"])
        slurm_out = posixpath.join(cfg.logs_dir, run_name + ".slurm.out")
        slurm_err = posixpath.join(cfg.logs_dir, run_name + ".slurm.err")
        log_paths = {"stdout": res["log_out"], "stderr": res["log_err"], "slurm_out": slurm_out, "slurm_err": slurm_err}
        save_job_state(res["jobid"], res["run_dir"], log_paths, extra={"notebook": "workflow-selector-cell1merged2"})

        globals().update(last_submit=res, jobid=res["jobid"], run_dir=res["run_dir"], log_out=res["log_out"], log_err=res["log_err"])
        with out_box:
            display(Markdown(f"Submitted — JOBID `{res['jobid']}`"))
            display(Markdown(f"RUN_DIR: `{res['run_dir']}`  \nLOGS: `{res['log_out']}`  `{res['log_err']}`"))
            display(Markdown(f"State saved to `~/.atomate2_jobs/{res['jobid']}.json` and to `{posixpath.join(cfg.logs_dir, 'job_'+res['jobid']+'.json')}`"))
    except Exception as e:
        with err_box:
            display(HTML("<div style='color:#b00020;white-space:pre-wrap'><b>Error:</b> {}</div>".format(e)))
    finally:
        btn.disabled=False

btn.on_click(_on_click)
display(ui)


<a id="cell-4"></a>

## 3) Monitor Job


Watches your SLURM job and shows the latest log lines.

- Set `WATCH = True` to auto-refresh every `INTERVAL` seconds.
- Uses `JOBID` from the submit cell; if missing, falls back to the latest run by `run_name`.
- Prints queue status plus tails of `.out` and `.err`.

You typically only tweak `WATCH` and `INTERVAL`.


In [14]:
# --- Cell 3: Fast SLURM Monitor (single cell, 20s default, 2s timeouts, sacct only at end) ---
# Requires from Cell 2: cfg, rmt
import base64, json, re, posixpath, shlex, time
import ipywidgets as w
from IPython.display import display, HTML
from tornado.ioloop import PeriodicCallback

# ============ helpers ============
def _run_remote(cmd: str):
    R = globals().get("rmt")
    if R is None:
        return (1, "", "rmt missing (run the connect cell)")
    rc, out, err = R.run(cmd, check=False, modules=False, export_env=False)
    return rc, (out or "").strip(), (err or "").strip()

def _strip_jobid(j): 
    return re.sub(r"\.(batch|extern)$", "", str(j or "").strip())

def _status_badge(summary):
    styles={"SUCCESS":("✅","#0a7f2e"),"RUNNING":("🏃‍♂️","#1f5eff"),
            "PENDING":("⏳","#8a6d00"),"FAILURE":("❌","#b00020"),"UNKNOWN":("❔","#444")}
    icon,color=styles.get(summary,styles["UNKNOWN"])
    return f"<span style='font-weight:700;color:{color}'>{icon} {summary}</span>"

def _classify(state, exitc):
    s = (state or "").upper()
    if s in {"PENDING","CONFIGURING","SUSPENDED","STAGE_OUT","RESV_DEL_HOLD"}: return "PENDING"
    if s in {"RUNNING","COMPLETING"}: return "RUNNING"
    if s.startswith("COMPLETED") and (not exitc or exitc.startswith("0:0")): return "SUCCESS"
    if s.startswith(("FAILED","CANCELLED","TIMEOUT","OUT_OF_MEMORY","NODE_FAIL","PREEMPTED")): return "FAILURE"
    if exitc and not exitc.startswith("0:0"): return "FAILURE"
    return "PENDING"

def _basename_from_stdout(stdout_path: str) -> str:
    if not stdout_path: return ""
    b = posixpath.basename(stdout_path)
    for suf in (".slurm.out", ".out"):
        if b.endswith(suf): return b[:-len(suf)]
    return ""

def _remote_dir_exists(p: str) -> bool:
    if not p: return False
    rc, out, _ = _run_remote(f"test -d {shlex.quote(p)} && echo YES || echo NO")
    return (out.strip() == "YES")

# ============ embedded Python watcher (uploaded via base64) ============
_WATCHER_PY = r"""
#!/usr/bin/env python3
import json, subprocess, time, sys, re, os

ENV = {"PATH": "/usr/bin:/bin:/usr/sbin:/sbin"}
TO = 2  # per-command timeout (seconds)

def run(cmd):
    try:
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
                             env=ENV, text=True, check=False, timeout=TO)
        return res.returncode, (res.stdout or "").strip()
    except subprocess.TimeoutExpired:
        return 124, ""
    except Exception:
        return 1, ""

def squeue_state(jid):
    rc, out = run(["/usr/bin/squeue","-j",jid,"-h","-o","%T"])
    if rc==0 and out: return out.splitlines()[0].strip()
    return ""

def scontrol_info(jid):
    rc, out = run(["/usr/bin/scontrol","show","job",jid])
    if rc!=0 or not out: return {"state":"", "stdout":"", "workdir":"", "jobname":""}
    def pick(rx):
        m = re.search(rx, out)
        return m.group(1) if m else ""
    return {
        "state":   pick(r"JobState=([^ \n]+)"),
        "stdout":  pick(r"StdOut=([^ \n]+)"),
        "workdir": pick(r"WorkDir=([^ \n]+)"),
        "jobname": pick(r"JobName=([^ \n]+)")
    }

def sacct_row(jid):
    rc, out = run(["/usr/bin/sacct","-X","-P","-n","-j",jid,"--format","JobIDRaw,State,ExitCode,JobName,StdOut,WorkDir"])
    if rc!=0 or not out: return {}
    wanted = (jid, f"{jid}.batch", f"{jid}.extern")
    for ln in out.splitlines()[:6]:
        parts = ln.split("|")
        if parts and parts[0] in wanted and len(parts)>=6:
            return {"state":parts[1],"exit":parts[2],"jobname":parts[3],"stdout":parts[4],"workdir":parts[5]}
    return {}

def sacct_brief(jid):
    rc, out = run(["/usr/bin/sacct","-X","-n","-P","-j",jid,"--format","JobID,JobName%30,State,Elapsed,Start,End,Partition%20"])
    return out if rc==0 else ""

def classify(state, exitc):
    s = (state or "").upper()
    if s in {"PENDING","CONFIGURING","SUSPENDED","STAGE_OUT","RESV_DEL_HOLD"}: return "PENDING"
    if s in {"RUNNING","COMPLETING"}: return "RUNNING"
    if s.startswith("COMPLETED") and (not exitc or exitc.startswith("0:0")): return "SUCCESS"
    if s.startswith(("FAILED","CANCELLED","TIMEOUT","OUT_OF_MEMORY","NODE_FAIL","PREEMPTED")): return "FAILURE"
    if exitc and not exitc.startswith("0:0"): return "FAILURE"
    return "PENDING"

def main():
    if len(sys.argv)<4:
        print("usage: watcher.py <jobid> <json_path> <interval_seconds>", file=sys.stderr); sys.exit(2)
    jid, json_path, interval = sys.argv[1], sys.argv[2], max(1, int(sys.argv[3]))

    # one-time static fields (no sacct)
    sc = scontrol_info(jid)
    st = squeue_state(jid) or sc.get("state","")
    stdout = sc.get("stdout",""); workdir=sc.get("workdir",""); jobname=sc.get("jobname","")
    exitc = ""

    while True:
        st_now = squeue_state(jid)
        if not st_now:
            st_now = scontrol_info(jid).get("state","")
        st = st_now or st or "UNKNOWN"

        # minimal brief to keep watcher light (no sacct while active)
        brief = f"{jid}|{st}"

        payload = {
            "state":  st,
            "exit":   exitc,
            "brief":  brief,
            "stdout": stdout,
            "workdir":workdir,
            "jobname":jobname,
            "ts": time.time()
        }
        tmp = json_path + ".tmp"
        with open(tmp,"w") as f: json.dump(payload, f)
        os.replace(tmp, json_path)

        if classify(st, exitc) in ("SUCCESS","FAILURE"):
            # final sacct once
            row = sacct_row(jid)
            if row:
                exitc  = row.get("exit","") or exitc
                stdout = row.get("stdout","") or stdout
                workdir= row.get("workdir","") or workdir
                jobname= row.get("jobname","") or jobname
            payload["exit"]  = exitc
            payload["brief"] = sacct_brief(jid)
            with open(tmp,"w") as f: json.dump(payload, f)
            os.replace(tmp, json_path)
            break

        time.sleep(interval)

if __name__ == "__main__":
    main()
"""

def _install_watcher_py(jid: str):
    base = f"/tmp/slurm_watch_{_strip_jobid(jid)}"
    pyf  = f"{base}.py"
    b64  = base64.b64encode(_WATCHER_PY.encode("utf-8")).decode("ascii")
    cmd = f"""bash --noprofile --norc -lc 'base64 -d > {shlex.quote(pyf)} <<B64
{b64}
B64
chmod +x {shlex.quote(pyf)} && echo OK {shlex.quote(pyf)}'"""
    rc, out, err = _run_remote(cmd)
    if rc != 0 or not out.startswith("OK"):
        return False, "", (err or out)
    return True, pyf, ""

def _ensure_remote_watcher(jobid: str, watch_interval_sec: int = 10):
    """Ensure one watcher process writing /tmp/slurm_watch_<jid>.json, launched with clean env."""
    jid = _strip_jobid(jobid)
    if not jid: return False, "empty jobid"
    base = f"/tmp/slurm_watch_{jid}"
    pidf, jsonf = f"{base}.pid", f"{base}.json"

    # Running?
    rc, out, err = _run_remote(
        f"""bash --noprofile --norc -lc 'if [ -f {shlex.quote(pidf)} ]; then p=$(cat {shlex.quote(pidf)} 2>/dev/null || true); if [ -n "$p" ] && kill -0 "$p" 2>/dev/null; then echo RUNNING {shlex.quote(jsonf)}; else echo STALE {shlex.quote(jsonf)}; fi; else echo NONE {shlex.quote(jsonf)}; fi'"""
    )
    need_start = not (rc == 0 and out.startswith("RUNNING"))

    ok, pyf, e = _install_watcher_py(jid)
    if not ok:
        return False, f"failed to install watcher script: {e}"

    if need_start:
        start_cmd = f"""bash --noprofile --norc -lc 'PATH=/usr/bin:/bin:/usr/sbin:/sbin nohup /usr/bin/python3 {shlex.quote(pyf)} {shlex.quote(jid)} {shlex.quote(jsonf)} {int(max(1, watch_interval_sec))} >/dev/null 2>&1 & echo $! > {shlex.quote(pidf)}; echo STARTED {shlex.quote(jsonf)}'"""
        rc2, out2, err2 = _run_remote(start_cmd)
        if rc2 != 0 or not out2.startswith("STARTED"):
            return False, f"failed to start watcher: {err2 or out2}"

    return True, jsonf

def _stop_remote_watcher(jobid: str):
    jid = _strip_jobid(jobid)
    if not jid: return
    base = f"/tmp/slurm_watch_{jid}"
    pidf = f"{base}.pid"
    _run_remote(f"""bash --noprofile --norc -lc 'if [ -f {shlex.quote(pidf)} ]; then p=$(cat {shlex.quote(pidf)} 2>/dev/null || true); [ -n "$p" ] && kill "$p" 2>/dev/null || true; rm -f {shlex.quote(pidf)}; fi'""")

def _read_watcher_json(json_path: str):
    rc, out, err = _run_remote(f"cat {shlex.quote(json_path)} 2>/dev/null")
    if rc != 0 or not out:
        return {}
    try:
        return json.loads(out)
    except Exception:
        return {}

# ============ UI (default 20s) ============
_prefill = {}
if isinstance(globals().get("last_submit"), dict): _prefill.update(globals()["last_submit"])
if globals().get("jobid"): _prefill.setdefault("jobid", globals()["jobid"])

txt_jobid   = w.Text(description="JobID:", value=str(_prefill.get("jobid","")), layout=w.Layout(width="260px"))
poll_every  = w.BoundedIntText(value=20, min=2, max=600, step=1, description="UI interval (s)")  # default 20s
btn_check   = w.Button(description="Check now", button_style="primary")
btn_start   = w.Button(description="Start polling")
btn_stop    = w.Button(description="Stop")
btn_killw   = w.Button(description="Kill watcher", tooltip="Stops the background watcher process on the cluster")

status_html = w.HTML("")
paths_html  = w.HTML("")

_cb = {"pc": None, "jobid": "", "json_path": "", "have_paths": False}

def _render(data):
    summary = _classify(data.get("state"), data.get("exit"))
    badge = _status_badge(summary)
    brief = data.get("brief","")
    ts = data.get("ts", None)
    age = f" &nbsp; <small style='color:#666'>(updated {max(0, int(time.time()-ts))}s ago)</small>" if isinstance(ts,(int,float)) else ""
    status_html.value = f"{badge}{age}<br><pre>{brief}</pre>"
    return summary

def _maybe_publish_paths(data, jobid):
    if _cb["have_paths"]: return
    flows_dir = getattr(globals().get("cfg", object()), "flows_dir", "")
    base = _basename_from_stdout(data.get("stdout","")) or (data.get("jobname","") or "")
    rd = posixpath.join(flows_dir.rstrip("/"), base) if (flows_dir and base) else ""
    published = []
    if rd and _remote_dir_exists(rd):
        globals()["run_dir"] = rd
        _cb["have_paths"] = True
        published.append(f"run_dir: <code>{rd}</code>")
    if data.get("stdout"):
        globals()["log_out"] = data["stdout"]
        published.append(f"log_out: <code>{data['stdout']}</code>")
    if jobid: globals()["jobid"] = jobid
    if published: paths_html.value = " • ".join(published)

def _tick():
    if not _cb["jobid"] or not _cb["json_path"]: return
    data = _read_watcher_json(_cb["json_path"])
    if not data: return
    summary = _render(data)
    _maybe_publish_paths(data, _cb["jobid"])
    if summary in ("SUCCESS","FAILURE"):
        _stop_polling()
        _stop_remote_watcher(_cb["jobid"])

def _start_polling(_=None):
    jid = _strip_jobid(txt_jobid.value)
    if not jid:
        status_html.value = "<span style='color:#b00'>Enter a JobID first.</span>"; return
    # watcher interval = half UI interval (min 5s)
    ok, path_or_err = _ensure_remote_watcher(jid, watch_interval_sec=max(5, int(poll_every.value//2)))
    if not ok:
        status_html.value = f"<span style='color:#b00'>Watcher error:</span> {path_or_err}"; return
    _cb.update({"jobid": jid, "json_path": path_or_err, "have_paths": False})
    data = _read_watcher_json(_cb["json_path"]) or {}
    if data:
        _render(data); _maybe_publish_paths(data, jid)
    if _cb["pc"]: _cb["pc"].stop()
    _cb["pc"] = PeriodicCallback(_tick, max(400, int(poll_every.value)*1000)); _cb["pc"].start()

def _stop_polling(_=None):
    if _cb["pc"]: _cb["pc"].stop(); _cb["pc"] = None

def _check_now(_=None):
    jid = _strip_jobid(txt_jobid.value)
    if not jid:
        status_html.value = "<span style='color:#b00'>Enter a JobID first.</span>"; return
    ok, path_or_err = _ensure_remote_watcher(jid, watch_interval_sec=10)
    if not ok:
        status_html.value = f"<span style='color:#b00'>Watcher error:</span> {path_or_err}"; return
    _cb.update({"jobid": jid, "json_path": path_or_err})
    data = _read_watcher_json(_cb["json_path"]) or {}
    status_html.value = "<span style='color:#888'>Waiting for watcher update…</span>" if not data else ""
    if data: _render(data); _maybe_publish_paths(data, jid)

def _kill_watcher(_=None):
    jid = _strip_jobid(txt_jobid.value)
    if jid:
        _stop_remote_watcher(jid)
        status_html.value = "<span style='color:#555'>Remote watcher stopped.</span>"

btn_check.on_click(_check_now)
btn_start.on_click(_start_polling)
btn_stop.on_click(_stop_polling)
btn_killw.on_click(_kill_watcher)

display(w.VBox([
    w.HTML("<h4>Monitor a SLURM job</h4>"),
    w.HBox([txt_jobid, btn_check]),
    w.HBox([poll_every, btn_start, btn_stop, btn_killw]),
    status_html,
    paths_html
]))


<a id="cell-5"></a>

## 4) Parse Job Output


Pulls a quick summary from the **latest run** (or the `run_dir` from submit):  
finds `vasprun.xml` / `OUTCAR` / `CONTCAR`, then prints JSON with **formula**, **natoms**, **energy**, **E-Fermi**, and **band gap** (if present).  
Also saves `last_contcar_path` for the visualization cell.

You usually don’t need to change anything here.


In [17]:
# === Cell 4: Parse (FAST) — Interactive Widget ===
import json, posixpath, time, textwrap
from collections import OrderedDict

import ipywidgets as w
from IPython.display import display, HTML, JSON as DJSON

# --- Preconditions: we rely on Cell 1 for rmt/_ensure_live_remote and on earlier cells for cfg/jobid/log paths ---
_ensure_rmt = globals().get("_ensure_live_remote")
rmt = _ensure_rmt() if callable(_ensure_rmt) else globals().get("rmt")
if rmt is None:
    raise RuntimeError("No remote session found. Run the connect cell first.")

cfg = globals().get("cfg")
if cfg is None:
    raise RuntimeError("Config object `cfg` not found.")

# Session hints from submit/monitor cells (optional)
_session_jobid = globals().get("jobid")
_explicit_log_out = globals().get("log_out")
_explicit_run_dir = globals().get("run_dir")

# Remote python resolver
_env_bin = globals().get("_env_bin") or (lambda cfg: (getattr(cfg, "remote_env_dir","") or "").rstrip("/") + "/bin")
remote_python = posixpath.join(_env_bin(cfg), "python")
flows_base = getattr(cfg, "flows_dir", None)
if not flows_base:
    raise RuntimeError("cfg.flows_dir is not set.")

# --------------------------
# Helpers
# --------------------------
def _recent_runs(base, limit=30):
    """Return recent run dirs under base, sorted by mtime desc (best effort)."""
    # Use shell for speed; fall back gently if errors
    cmd = f"""bash -lc 'shopt -s nullglob dotglob; for d in "{base}"/*; do [[ -d "$d" ]] && echo "$(stat -c %Y "$d" 2>/dev/null || stat -f %m "$d") $d"; done | sort -nr | head -n {int(limit)}'"""
    rc, out, err = rmt.run(cmd, check=False, modules=False, export_env=False)
    items = []
    if out:
        for line in out.strip().splitlines():
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                items.append(parts[1].strip())
    return items

def _compose_remote_parser(run_dir, parse_eigs, max_depth, jobid_hint, log_hint):
    py = f'''
import os, sys, json, gzip, re

BASE      = {flows_base!r}
RD_EXPL   = {run_dir!r}
LOG_EXPL  = {log_hint!r}
JOBID     = {repr(jobid_hint)}
MAX_DEPTH = int(os.environ.get("PARSER_MAX_DEPTH", "1"))
PARSE_EIG = bool(int(os.environ.get("PARSER_PARSE_EIGS", "0")))

def newest_run_dir(base):
    try:
        ds=[os.path.join(base,d) for d in os.listdir(base)]
        ds=[d for d in ds if os.path.isdir(d)]
        return max(ds, key=lambda p: os.path.getmtime(p)) if ds else None
    except Exception:
        return None

def _depth(p): 
    p2=p.rstrip(os.sep); 
    return 0 if not p2 else p2.count(os.sep)

def find_file(run_dir, name):
    # quick paths
    for p in (os.path.join(run_dir,name), os.path.join(run_dir,name+".gz"),
              os.path.join(run_dir,"latest_calc",name), os.path.join(run_dir,"latest_calc",name+".gz")):
        if os.path.exists(p): 
            return p
    # bounded walk
    base=_depth(run_dir); best=None; bestt=-1.0
    for root, dirs, files in os.walk(run_dir):
        if _depth(root)-base >= MAX_DEPTH: 
            dirs[:]=[]
        for cand in (name, name+".gz"):
            if cand in files:
                p=os.path.join(root,cand)
                try:
                    t=os.path.getmtime(p)
                    if t>bestt: 
                        best, bestt=p, t
                except Exception: 
                    pass
    return best

def open_text_auto(path):
    if not path: 
        return None
    return (gzip.open(path,"rt",errors="ignore") if path.endswith(".gz") else open(path,"rt",errors="ignore"))

def read_first_lines(path, n=160):
    f=open_text_auto(path); 
    if not f: return ""
    try:
        return "".join([next(f,"") for _ in range(n)])
    except Exception:
        return ""
    finally:
        try: f.close()
        except Exception: pass

def read_last_bytes(path, max_bytes=40000):
    if not path or not os.path.exists(path) or path.endswith(".gz"): 
        return ""
    try:
        with open(path,"rb") as f:
            f.seek(0,2); size=f.tell(); f.seek(max(0,size-max_bytes))
            return f.read().decode("utf-8","ignore")
    except Exception:
        return ""

def parse_kpoints_mesh(path):
    if not path: return None
    L=read_first_lines(path,20).splitlines()
    if len(L)<4: return None
    try:
        nums=[int(x) for x in L[3].split()[:3]]
        return tuple(nums) if len(nums)==3 else None
    except Exception:
        return None

def guess_potcar_functional(path):
    if not path: return None
    head=read_first_lines(path,40)
    base="LDA" if ("PAW_LDA" in head or "LDA" in head) else ("PBE" if ("PAW_PBE" in head or "GGA" in head) else None)
    p=(path or "").lower()
    if "pbe_64" in p: return "PBE_64"
    if "pbe_54" in p: return "PBE_54"
    if "pbe_52" in p: return "PBE_52"
    return base

def parse_outcar_quick(path):
    E=EF=None
    f=open_text_auto(path)
    if not f: return E,EF
    try:
        for line in f:
            if E is None and "free  energy   TOTEN" in line:
                try: E=float(line.split()[-2])
                except Exception: pass
            if EF is None and "E-fermi" in line:
                t=line.split()
                if len(t)>=3:
                    try: EF=float(t[2])
                    except Exception: pass
            if E is not None and EF is not None: break
    except Exception: 
        pass
    finally:
        try: f.close()
        except Exception: pass
    return E,EF

def parse_outcar_tail_extras(path):
    tail=read_last_bytes(path,60000)
    if not tail: return None,None,None
    converged=bool(re.search(r"(reached required accuracy|aborting loop because EDIFF is reached)", tail, re.I))
    msteps=re.findall(r"external pressure|POSITION\\s+TOTAL-FORCE", tail)
    ionic=len(msteps) if msteps else None
    mm=re.search(r"number of electron\\s*=\\s*.*?magnetization\\s*=\\s*([-.\\d]+)", tail, re.I|re.S) or re.search(r"tot\\s+magnetization\\s*=\\s*([-.\\d]+)", tail, re.I)
    mag=float(mm.group(1)) if mm else None
    return converged, ionic, mag

def parse_poscar_formula_natoms(path):
    if not path or not os.path.exists(path): return None,None
    try:
        with open_text_auto(path) as f:
            L=[ln.strip() for ln in [next(f,"") for _ in range(16)] if ln]
        if len(L)<7: return None,None
        t6=L[5].split()
        is_syms=lambda ts: all(t.isalpha() for t in ts)
        if is_syms(t6):
            syms=t6; cnt=[int(x) for x in L[6].split()]
        else:
            syms=[]; cnt=[int(x) for x in t6]
        nat=sum(cnt) if cnt else None
        form=None
        if syms and len(syms)==len(cnt):
            form="".join([el+(str(n) if n!=1 else "") for el,n in zip(syms,cnt)])
        return form, nat
    except Exception:
        return None,None

def parse_workflow_from_log(path):
    if not path or not os.path.exists(path): return None
    try:
        with open(path,"rt",errors="ignore") as f:
            tail=f.read()[-4000:]
        m=re.search(r"Starting job\\s*-\\s*(\\w+)", tail)
        return m.group(1) if m else None
    except Exception:
        return None

# --- choose run dir ---
rd = RD_EXPL or newest_run_dir(BASE)
if not rd:
    print(json.dumps({{"error":"no_run_dir","hint":f"No runs under {{BASE}}","base":BASE}})); sys.exit(0)

latest_calc=os.path.join(rd,"latest_calc")
calc_dir=os.path.realpath(latest_calc) if os.path.exists(latest_calc) else rd

# --- pick files (gz or plain, shallow bounded) ---
vasp_path   = find_file(rd,"vasprun.xml")
outcar_path = find_file(rd,"OUTCAR")
contcar_path= find_file(rd,"CONTCAR")
poscar_path = find_file(rd,"POSCAR") if not contcar_path else None
kpoints_path= find_file(rd,"KPOINTS")
potcar_path = find_file(rd,"POTCAR") or find_file(rd,"POTCAR.orig")

# --- FAST summary ---
E=EF=BG=None; formula=natoms=None; Epa=None
kmesh=parse_kpoints_mesh(kpoints_path)
pot_guess=guess_potcar_functional(potcar_path)
workflow=parse_workflow_from_log(LOG_EXPL)
converged=ionic_steps=mag_tot=None

if outcar_path:
    e,ef=parse_outcar_quick(outcar_path); 
    if e is not None: E=e
    if ef is not None: EF=ef
    c,i,m=parse_outcar_tail_extras(outcar_path)
    converged = c if c is not None else converged
    ionic_steps = i if i is not None else ionic_steps
    mag_tot = m if m is not None else mag_tot

for p in filter(None, (contcar_path, poscar_path)):
    if (formula is None or natoms is None) and os.path.exists(p):
        f2,n2 = parse_poscar_formula_natoms(p)
        if formula is None and f2 is not None: formula=f2
        if natoms  is None and n2 is not None: natoms=n2
        if formula is not None and natoms is not None: break

need_vr = PARSE_EIG or (E is None or formula is None or natoms is None)
if need_vr and vasp_path and os.path.exists(vasp_path):
    try:
        from pymatgen.io.vasp.outputs import Vasprun
        try:
            vr = Vasprun(vasp_path, parse_dos=False, parse_eigenvalues=PARSE_EIG, exception_on_bad_xml=False, parse_potcar_file=False)
        except TypeError:
            vr = Vasprun(vasp_path, parse_dos=False, parse_eigen=PARSE_EIG, exception_on_bad_xml=False, parse_potcar_file=False)
        if E is None:
            fe=getattr(vr,"final_energy",None); 
            if fe is not None: E=float(fe)
        if EF is None:
            ef=getattr(vr,"efermi",None); 
            if ef is not None: EF=float(ef)
        if formula is None or natoms is None:
            fs=getattr(vr,"final_structure",None)
            if fs is not None:
                if natoms is None: natoms=len(fs)
                if formula is None:
                    try: formula="".join(fs.composition.formula.split())
                    except Exception: pass
        try:
            if converged is None: converged=bool(getattr(vr,"converged",False))
        except Exception: pass
        if PARSE_EIG:
            try:
                ebp=getattr(vr,"eigenvalue_band_properties")
                if callable(ebp): gap, cbm, vbm, _=ebp()
                else: gap, cbm, vbm, _=ebp
                BG=float(gap)
            except Exception:
                try:
                    bs=vr.get_band_structure(); gap=bs.get_band_gap().get("energy")
                    if gap is not None: BG=float(gap)
                except Exception: pass
    except Exception as e:
        print("Vasprun parse skipped/error:", str(e), file=sys.stderr)

if E is not None and natoms:
    try: Epa=E/float(natoms)
    except Exception: pass

summary = {{
    "jobid": JOBID, "dir": rd, "calc_dir": calc_dir, "log_out": LOG_EXPL,
    "vasprun_path": vasp_path, "outcar_path": outcar_path, "contcar_path": contcar_path, "poscar_path": poscar_path,
    "kpoints_path": kpoints_path, "potcar_path": potcar_path, "workflow": workflow,
    "formula": formula, "natoms": natoms, "kmesh": kmesh, "potcar_functional_guess": pot_guess,
    "energy_ev": E, "energy_per_atom_ev": Epa, "efermi_ev": EF, "band_gap_ev": BG,
    "converged": converged, "ionic_steps": ionic_steps, "total_magnetization_bohr": mag_tot
}}
print(json.dumps(summary, indent=2))
'''
    env = f'PARSER_PARSE_EIGS={int(parse_eigs)} PARSER_MAX_DEPTH={int(max_depth)}'
    cmd = f"""{env} "{remote_python}" - <<'PY'\n{py}\nPY\n"""
    return cmd

def _render_table(info: dict) -> HTML:
    keys = [
        ("jobid", "Job ID"),
        ("workflow", "Workflow"),
        ("dir", "Run Dir"),
        ("calc_dir", "Calc Dir"),
        ("formula", "Formula"),
        ("natoms", "Atoms"),
        ("kmesh", "K-pts"),
        ("potcar_functional_guess", "POTCAR"),
        ("energy_ev", "Energy (eV)"),
        ("energy_per_atom_ev", "E/atom (eV)"),
        ("efermi_ev", "E_F (eV)"),
        ("band_gap_ev", "Band gap (eV)"),
        ("converged", "Converged"),
        ("ionic_steps", "Ionic steps"),
        ("total_magnetization_bohr", "Mag (μB)"),
    ]

    MISSING = "<span style='color:#777'>—</span>"

    def fmt(v):
        if v is None or v == "":
            return MISSING
        if isinstance(v, float):
            return f"{v:.6g}"
        if isinstance(v, bool):
            return "✅" if v else "❌"
        if isinstance(v, (list, tuple)):
            try:
                return "×".join(str(int(x)) for x in v)
            except Exception:
                return str(v)
        return str(v)

    rows = []
    for k, label in keys:
        cell = fmt(info.get(k))
        rows.append(
            f"<tr>"
            f"<th style='text-align:left;padding-right:10px'>{label}</th>"
            f"<td style='text-align:left'>{cell}</td>"
            f"</tr>"
        )

    html = "<table style='border-collapse:collapse'>" + "".join(rows) + "</table>"
    return HTML(html)

# --------------------------
# Widgets
# --------------------------
lbl = w.HTML("<b>Parse VASP results (FAST)</b>")
status = w.HTML("")

auto_token = "__AUTO_NEWEST__"
recent_opts = [(f"auto (newest in {flows_base})", auto_token)]
try:
    for d in _recent_runs(flows_base, limit=30):
        recent_opts.append((d, d))
except Exception:
    pass
# If a prior run_dir was set, prefer it
if _explicit_run_dir and _explicit_run_dir not in [v for _, v in recent_opts]:
    recent_opts.insert(1, (_explicit_run_dir, _explicit_run_dir))

run_dir_dd = w.Dropdown(options=recent_opts, value=_explicit_run_dir or auto_token, layout=w.Layout(width='100%'))
refresh_btn = w.Button(description="Refresh", icon="refresh")
eigs_tb = w.ToggleButton(value=bool(int(globals().get("PARSE_EIGS", 0))), description="Parse eigenvalues (band gap)", icon="wave-square")
depth_slider = w.IntSlider(value=int(globals().get("PARSE_MAX_DEPTH", 1)), min=0, max=4, step=1, description="Max depth")
save_cb = w.Checkbox(value=True, description="Save summary to run dir (parse_summary.json)")
parse_btn = w.Button(description="Parse", button_style="primary", icon="play")

summary_out = w.Output()
json_out = w.Output(layout=w.Layout(max_height="320px", overflow="auto", border="1px solid #ddd"))

# --------------------------
# Actions
# --------------------------
def _do_refresh(_=None):
    status.value = "Scanning recent runs…"
    opts = [(f"auto (newest in {flows_base})", auto_token)]
    try:
        for d in _recent_runs(flows_base, limit=30):
            opts.append((d, d))
    except Exception as e:
        status.value = f"<span style='color:#c00'>Refresh failed: {e}</span>"
        return
    # keep current value if still present
    cur = run_dir_dd.value
    run_dir_dd.options = opts
    if any(v == cur for _, v in opts):
        run_dir_dd.value = cur
    else:
        run_dir_dd.value = auto_token
    status.value = "Run list refreshed."

def _do_parse(_=None):
    start = time.perf_counter()
    status.value = "Running parser…"
    summary_out.clear_output()
    json_out.clear_output()
    # Decide run_dir
    chosen = run_dir_dd.value
    rd_for_remote = None if chosen == auto_token else chosen
    # Compose and run remote
    cmd = _compose_remote_parser(
        run_dir=rd_for_remote,
        parse_eigs=eigs_tb.value,
        max_depth=depth_slider.value,
        jobid_hint=_session_jobid,
        log_hint=_explicit_log_out
    )
    rc, out, err = rmt.run(cmd, check=False, modules=False, export_env=False)

    try:
        info = json.loads(out) if out and out.strip() else {"error":"no_output","stderr":(err or "").strip()}
    except Exception:
        info = {"error":"bad_json","raw_out":(out or "")[:1000], "stderr":(err or "").strip()}

    # Optionally save summary back to run dir
    if save_cb.value and isinstance(info, dict):
        try:
            rd = info.get("dir") or rd_for_remote or flows_base
            summary_path = posixpath.join(rd, "parse_summary.json")
            payload = json.dumps(info, indent=2, ensure_ascii=False)
            save_cmd = f"cat > '{summary_path}' <<'JSON'\n{payload}\nJSON\n"
            rmt.run(save_cmd, check=False, modules=False, export_env=False)
            saved_msg = f"Saved summary → <code>{summary_path}</code>"
        except Exception as e:
            saved_msg = f"<span style='color:#c00'>Save failed:</span> {e}"
    else:
        saved_msg = ""

    # Render outputs
    with summary_out:
        if "error" in info:
            display(HTML(f"<b style='color:#c00'>Error:</b> {info.get('error')}"))
            if info.get("stderr"):
                display(HTML(f"<pre style='white-space:pre-wrap;color:#a00'>{info['stderr']}</pre>"))
        else:
            display(_render_table(info))
        if saved_msg:
            display(HTML(saved_msg))
        elapsed = time.perf_counter() - start
        display(HTML(f"⏱️ <i>Elapsed:</i> {elapsed:.2f} s"))

    with json_out:
        display(DJSON(info))

    status.value = "Done."

refresh_btn.on_click(_do_refresh)
parse_btn.on_click(_do_parse)

# Initial draw
ui = w.VBox([
    lbl,
    w.HBox([w.Label("Run:"), run_dir_dd, refresh_btn]),
    w.HBox([eigs_tb, depth_slider, save_cb]),
    w.HBox([parse_btn]),
    status,
    w.HTML("<hr>"),
    w.HTML("<b>Summary</b>"),
    summary_out,
    w.HTML("<b>Raw JSON</b>"),
    json_out
])
display(ui)


<a id="cell-6"></a>

## 5) Visualize Structure

Fetches the latest **CONTCAR** (or POSCAR) on the cluster via SSH, analyzes it with **pymatgen** to get a conventional cell, and returns a compressed **CIF** back to the notebook for rendering with **py3Dmol**.

- Prefers the `last_contcar_path` set by the Parse cell if available; otherwise searches under the run directory (root → `latest_calc/` → bounded nested walk).  
- Caches results on the remote side (`.viz_cache/symm.json` by default) keyed by file mtime+size and supercell choice, so repeated runs are fast.  
- Applies symmetry analysis with configurable tolerances (`cfg.viz_symprec`, `cfg.viz_angle_tolerance`) and converts to a conventional standard cell.  
- Optionally builds a supercell (default **2×2×2**) or leaves the conventional cell as-is, controlled by `cfg.viz_force_supercell` and `cfg.viz_supercell`.  
- Sends the CIF back in **gzip+base64** form to minimize network transfer.  
- Locally renders with **py3Dmol**, adding the unit cell outline and using sticks-only style for speed.  

### Configurable parameters 

- `cfg.viz_force_supercell` (bool, default `True`) → whether to expand the structure.  
- `cfg.viz_supercell` (tuple of 3 ints, default `(2,2,2)`) → supercell size if forced.  
- `cfg.viz_symprec` (float, default `1e-3`) → symmetry tolerance.  
- `cfg.viz_angle_tolerance` (float, default `5.0`) → angle tolerance for symmetry finder.  
- `cfg.viz_width`, `cfg.viz_height` (ints, default `800×550`) → rendering canvas size.  
- `cfg.viz_stick_radius` (float, default `0.14`) → stick thickness in rendering.  
- `cfg.viz_max_depth` (int, default `2`) → maximum subdirectory depth to search under `run_dir`.  
- `cfg.viz_cache_dirname` (str, default `".viz_cache"`) → name of the remote cache folder.  

These knobs give you full control over performance vs. detail without touching the cell itself. By default, everything runs safely and efficiently for typical VASP outputs.


In [18]:
# --- Structure Visualization (REMOTE-ONLY, fast + flat UI, scratch I/O, inline small transfer) ---
import os, sys, time, json, gzip, base64, posixpath
import ipywidgets as widgets
from IPython.display import display, clear_output

# Prereqs from the connection / earlier cells
try:
    cfg, run_dir, rmt
except NameError:
    raise RuntimeError("Run the connection cell first (needs cfg, rmt, run_dir).")

# ===== Defaults from your globals so it stays in sync =====
_g = lambda k, d: globals().get(k, d)
VIZ_DEFAULTS = dict(
    VIZ_USE_PATH         = _g("last_contcar_path", None),
    VIZ_FORCE_SUPERCELL  = bool(_g("VIZ_FORCE_SUPERCELL", True)),
    VIZ_SUPERCELL        = tuple(_g("VIZ_SUPERCELL", (2,2,2))),
    VIZ_CELL_MODE        = str(_g("VIZ_CELL_MODE", "conventional")).lower(),  # conventional|primitive|input
    VIZ_SYMPREC          = float(_g("VIZ_SYMPREC", 1e-3)),
    VIZ_ANGLE_TOL        = float(_g("VIZ_ANGLE_TOL", 5.0)),
    VIZ_SHOW_CELL        = bool(_g("VIZ_SHOW_CELL", True)),
    VIZ_STYLE            = str(_g("VIZ_STYLE", "sticks")).lower(),            # sticks|ballstick|spheres|lines
    VIZ_WIDTH            = int(_g("VIZ_WIDTH", 800)),
    VIZ_HEIGHT           = int(_g("VIZ_HEIGHT", 550)),
    VIZ_STICK_RADIUS     = float(_g("VIZ_STICK_RADIUS", 0.14)),
    VIZ_AUTO_LIMIT_ATOMS = int(_g("VIZ_AUTO_LIMIT_ATOMS", 3500)),
)

# ===== UI (flat, readable labels; no accordions) =====
LABEL_W = "170px"
FIELD_W = "80%"

def _style(w):
    w.style = {"description_width": LABEL_W}
    w.layout = widgets.Layout(width=FIELD_W)

w_header = widgets.HTML("<b>Structure Visualization (Remote)</b>")

w_path  = widgets.Text(value=VIZ_DEFAULTS["VIZ_USE_PATH"] or "", description="Use path (opt)")
w_mode  = widgets.Dropdown(options=["conventional","primitive","input"], value=VIZ_DEFAULTS["VIZ_CELL_MODE"], description="Cell mode")
w_sym   = widgets.FloatText(value=VIZ_DEFAULTS["VIZ_SYMPREC"], description="symprec")
w_ang   = widgets.FloatText(value=VIZ_DEFAULTS["VIZ_ANGLE_TOL"], description="angle tol")
w_force = widgets.Checkbox(value=VIZ_DEFAULTS["VIZ_FORCE_SUPERCELL"], description="Force supercell")
w_scx   = widgets.BoundedIntText(value=VIZ_DEFAULTS["VIZ_SUPERCELL"][0], min=1, max=12, step=1, description="a×")
w_scy   = widgets.BoundedIntText(value=VIZ_DEFAULTS["VIZ_SUPERCELL"][1], min=1, max=12, step=1, description="b×")
w_scz   = widgets.BoundedIntText(value=VIZ_DEFAULTS["VIZ_SUPERCELL"][2], min=1, max=12, step=1, description="c×")
w_show  = widgets.Checkbox(value=VIZ_DEFAULTS["VIZ_SHOW_CELL"], description="Show cell")
w_style = widgets.Dropdown(options=["sticks","ballstick","spheres","lines"], value=VIZ_DEFAULTS["VIZ_STYLE"], description="Style")
w_w     = widgets.IntText(value=VIZ_DEFAULTS["VIZ_WIDTH"], description="Width")
w_h     = widgets.IntText(value=VIZ_DEFAULTS["VIZ_HEIGHT"], description="Height")
w_rad   = widgets.FloatText(value=VIZ_DEFAULTS["VIZ_STICK_RADIUS"], description="Stick radius")

for w in (w_path, w_mode, w_sym, w_ang, w_force, w_scx, w_scy, w_scz, w_show, w_style, w_w, w_h, w_rad):
    _style(w)

btn_quick  = widgets.Button(description="Quick pick (run_dir/latest_calc)", icon="magic")
btn_render = widgets.Button(description="Render (remote)", button_style="success", icon="play")
lbl        = widgets.HTML()

out_info   = widgets.Output(layout={"border":"1px solid #eee"})
out_view   = widgets.Output()

# Flat stack, like your Cell 2 form (no hidden sections)
ui = widgets.VBox([
    w_header,
    btn_quick,
    w_path,
    w_mode,
    w_sym,
    w_ang,
    w_force,
    widgets.HBox([w_scx, w_scy, w_scz]),
    w_show,
    w_style,
    w_w,
    w_h,
    w_rad,
    widgets.HBox([btn_render, lbl]),
    out_info,
    out_view
])
display(ui)

# ===== helpers =====
INLINE_MAX_BYTES = 5 * 1024 * 1024   # inline return threshold for CIF.gz
REMOTE_TIMEOUT = 300
remote_python = posixpath.join((cfg.remote_env_dir or "").rstrip("/"), "bin/python").replace("\\","/")

def _persist_globals():
    globals().update(
        VIZ_FORCE_SUPERCELL = bool(w_force.value),
        VIZ_SUPERCELL = (int(w_scx.value), int(w_scy.value), int(w_scz.value)),
        VIZ_CELL_MODE = str(w_mode.value),
        VIZ_SYMPREC = float(w_sym.value),
        VIZ_ANGLE_TOL = float(w_ang.value),
        VIZ_SHOW_CELL = bool(w_show.value),
        VIZ_STYLE = str(w_style.value),
        VIZ_WIDTH = int(w_w.value),
        VIZ_HEIGHT = int(w_h.value),
        VIZ_STICK_RADIUS = float(w_rad.value),
        last_contcar_path = (w_path.value or None),
    )

def _quick_pick():
    rd = run_dir.replace("\\","/")
    checks = []
    if w_path.value.strip():
        checks.append(w_path.value.strip())
    for d in ("", "latest_calc"):
        for n in ("CONTCAR","POSCAR","CONTCAR.gz","POSCAR.gz"):
            checks.append(posixpath.join(rd, d, n))
    test = " || ".join([f'(test -f "{p}" && echo "{p}")' for p in checks])
    rc, out, err = rmt.run(test + " || true", check=False)
    lines = [ln.strip() for ln in (out or "").splitlines() if ln.strip()]
    if lines:
        w_path.value = lines[0]
        return True
    return False

def _stream_remote(cmd: str, timeout: int = REMOTE_TIMEOUT):
    """Run a remote command with live streaming + hard timeout. Returns (rc, out, err, elapsed)."""
    transport = rmt.client.get_transport()
    if not (transport and transport.is_active()):
        raise RuntimeError("SSH transport inactive.")
    t0 = time.perf_counter()
    chan = transport.open_session()
    chan.settimeout(5.0)
    chan.exec_command(cmd)
    out_chunks, err_chunks = [], []
    while True:
        if chan.recv_ready():
            data = chan.recv(4096)
            if data:
                sys.stdout.write(data.decode("utf-8","ignore")); sys.stdout.flush()
                out_chunks.append(data)
        if chan.recv_stderr_ready():
            data = chan.recv_stderr(4096)
            if data:
                sys.stderr.write(data.decode("utf-8","ignore")); sys.stderr.flush()
                err_chunks.append(data)
        if chan.exit_status_ready(): break
        if time.perf_counter() - t0 > timeout:
            try: chan.close()
            except Exception: pass
            raise TimeoutError(f"[remote] timeout after {timeout}s")
        time.sleep(0.03)
    rc = chan.recv_exit_status()
    return rc, b"".join(out_chunks).decode("utf-8","ignore"), b"".join(err_chunks).decode("utf-8","ignore"), time.perf_counter()-t0

def _render_remote_inline():
    out_info.clear_output(); out_view.clear_output()
    lbl.value = "⏳ rendering…"
    _persist_globals()

    target = w_path.value.strip() or None
    if not target:
        if not _quick_pick():
            lbl.value = "<b style='color:darkorange'>⚠️ No CONTCAR/POSCAR at run_dir/latest_calc</b>"
            return
        target = w_path.value.strip()

    heredoc = f"""PYTHONDONTWRITEBYTECODE=1 \\
"{remote_python}" -u -B - <<'PY'
import os, sys, json, time, gzip, base64, socket, platform
from pymatgen.io.vasp import Poscar
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

target={target!r}
cell_mode={w_mode.value!r}
symprec=float({float(w_sym.value)})
angle_tol=float({float(w_ang.value)})
force_super=bool({bool(w_force.value)})
super_tuple=({int(w_scx.value)},{int(w_scy.value)},{int(w_scz.value)})
atom_limit=3500
INLINE_MAX={int(INLINE_MAX_BYTES)}

base = os.environ.get("SLURM_TMPDIR") or os.path.join("/tmp", os.environ.get("USER","viz"))
cache_dir = os.path.join(base, "viz_one_shot"); os.makedirs(cache_dir, exist_ok=True)
cif_gz = os.path.join(cache_dir, "structure.cif.gz")

# Load & process
t=time.monotonic()
if target.endswith(".gz"):
    with gzip.open(target,"rt",errors="ignore") as f:
        s = Poscar.from_string(f.read()).structure
else:
    s = Poscar.from_file(target).structure
t_load=time.monotonic()-t

t=time.monotonic()
try: sGA=SpacegroupAnalyzer(s, symprec=symprec, angle_tolerance=angle_tol)
except Exception: sGA=None
if   cell_mode=="conventional" and sGA:
    base_struct=sGA.get_conventional_standard_structure()
elif cell_mode=="primitive" and sGA:
    base_struct=sGA.get_primitive_standard_structure()
else:
    base_struct=s
t_sym=time.monotonic()-t

def n_after(d): return len(base_struct)*max(1,d[0])*max(1,d[1])*max(1,d[2])
tgt=list(super_tuple if force_super else (1,1,1))
if atom_limit>0 and n_after(tgt)>atom_limit:
    while n_after(tgt)>atom_limit and any(x>1 for x in tgt):
        i=max(range(3), key=lambda k: tgt[k])
        if tgt[i]>1: tgt[i]-=1
        else: break

t=time.monotonic()
try: view = base_struct * tuple(tgt)
except Exception:
    sc=tuple(max(1,int(x)) for x in tgt)
    view=base_struct.copy(); view.make_supercell([[sc[0],0,0],[0,sc[1],0],[0,0,sc[2]]])
t_super=time.monotonic()-t

t=time.monotonic(); cif=view.to(fmt="cif"); t_cif=time.monotonic()-t
t=time.monotonic()
with gzip.open(cif_gz,"wb") as f: f.write(cif.encode())
gz_bytes=os.path.getsize(cif_gz); t_gz=time.monotonic()-t

sg=None
if sGA:
    try: sg=sGA.get_space_group_symbol()
    except Exception: pass

b64=None
if gz_bytes <= INLINE_MAX:
    with open(cif_gz,"rb") as f: b64 = base64.b64encode(f.read()).decode()

out={{"path": target, "cell_mode": cell_mode, "space_group": sg,
     "natoms_base_cell": len(base_struct), "natoms_rendered": len(view),
     "supercell_tuple": list(tgt), "remote_cif_gz": (None if b64 else cif_gz),
     "_gz_bytes": gz_bytes, "_inline": bool(b64),
     "_timings": {{"load": t_load, "symmetry": t_sym, "supercell": t_super, "to_cif": t_cif, "gzip_write": t_gz}},
     "_sys": {{"hostname": os.uname().nodename, "python": platform.python_version(), "scratch": cache_dir}}}}

print("JSON:"+json.dumps(out), flush=True)
if b64: print("B64:"+b64, flush=True)

import os as _os
_os._exit(0)
PY
"""
    try:
        rc, out, err, elapsed = _stream_remote(heredoc, timeout=300)
    except TimeoutError as e:
        lbl.value = "<b style='color:darkorange'>⚠️ remote timeout</b>"
        with out_info: print(e)
        return

    if rc != 0 or not out.strip():
        lbl.value = "<b style='color:darkorange'>⚠️ remote error</b>"
        with out_info: print(err or "(no stderr)")
        return

    meta, b64 = None, None
    for line in out.splitlines():
        if line.startswith("JSON:"): meta = json.loads(line[5:])
        elif line.startswith("B64:"): b64 = line[4:].strip()
    if not meta:
        lbl.value = "<b style='color:darkorange'>⚠️ parse error</b>"
        with out_info: print("Raw stdout tail:\n", "\n".join(out.splitlines()[-40:]))
        return

    # Get CIF text (inline small; SFTP fallback for large)
    if meta.get("_inline") and b64:
        cif_text = gzip.decompress(base64.b64decode(b64)).decode("utf-8","ignore")
        transfer_note = "inline"
    else:
        with rmt.client.open_sftp() as sftp:
            local_cif_gz = "structure.cif.gz"
            sftp.get(meta["remote_cif_gz"], local_cif_gz)
        with gzip.open(local_cif_gz, "rt", encoding="utf-8", errors="ignore") as f:
            cif_text = f.read()
        transfer_note = f"SFTP ({meta.get('_gz_bytes')} bytes)"

    # info
    with out_info:
        clear_output()
        sc = meta.get("supercell_tuple",(1,1,1))
        tm = meta.get("_timings", {})
        print("Source file:", meta.get("path"))
        print("Cell mode:", meta.get("cell_mode"), "Space group:", meta.get("space_group"))
        print(f"Atoms (base): {meta.get('natoms_base_cell')}  Atoms rendered ({sc[0]}x{sc[1]}x{sc[2]}): {meta.get('natoms_rendered')}")
        print("Remote timings (s): load={:.3f}, symmetry={:.3f}, supercell={:.3f}, to_cif={:.3f}, gzip={:.3f}".format(
            tm.get("load",0), tm.get("symmetry",0), tm.get("supercell",0), tm.get("to_cif",0), tm.get("gzip_write",0)
        ))
        print("Transfer:", transfer_note)

    # render
    try:
        import py3Dmol
    except Exception:
        lbl.value = "<b style='color:darkorange'>⚠️ py3Dmol not installed</b>"
        with out_info:
            print("Install once with: pip install py3Dmol")
        return

    with out_view:
        clear_output()
        v = py3Dmol.view(width=int(w_w.value), height=int(w_h.value))
        v.setBackgroundColor("white")
        v.addModel(cif_text, "cif")
        if bool(w_show.value): v.addUnitCell()
        if   w_style.value=="ballstick": v.setStyle({}, {"stick":{"radius":float(w_rad.value)}, "sphere":{"scale":0.3}})
        elif w_style.value=="spheres":   v.setStyle({}, {"sphere":{"scale":0.45}})
        elif w_style.value=="lines":     v.setStyle({}, {"line":{}})
        else:                             v.setStyle({}, {"stick":{"radius":float(w_rad.value)}})
        v.zoomTo()
        display(v)

    lbl.value = "✅ done"

def _on_quick(_):
    lbl.value = "⏳ picking…"
    ok = _quick_pick()
    lbl.value = "✅ picked" if ok else "<b style='color:darkorange'>⚠️ nothing obvious</b>"

def _on_render(_):
    btn_render.disabled = True
    try:
        _render_remote_inline()
    finally:
        btn_render.disabled = False

btn_quick.on_click(_on_quick)
btn_render.on_click(_on_render)


<a id="cell-7"></a>
## 6. Plot band structure (optional)

Tries to plot a band structure - use only if you parsed a succesful band structure chain.

In [21]:
# --- Cell: Bandstructure Plotter (manual Save PNG + auto remote sync/copy, warnings silenced, red dashed ↓ spin) ---
# - Uses BSPlotter.get_plot for identical axes/ticks
# - Spin ↓ is explicitly colored red and dashed so both channels are visible
# - Works with remote bands dir (via `rmt`) and manual Save PNG + optional remote copy

import os, io, time, base64, posixpath, shlex, gzip, hashlib, ipywidgets as w
import warnings
from pathlib import Path
from IPython.display import display, Markdown

from pymatgen.io.vasp.outputs import Vasprun
from pymatgen.electronic_structure.plotter import BSPlotter
from pymatgen.electronic_structure.core import Spin

# ====== UI ======
_default_bands_dir = ""
try:
    _rd = globals().get("run_dir", "")
    if _rd:
        _default_bands_dir = posixpath.join(_rd, "02_bands_gw_true")
except Exception:
    pass

bands_dir_tb = w.Text(value=_default_bands_dir, description="Bands dir", layout={"width":"650px"})
zero_ref_dd  = w.Dropdown(options=[("VBM (zero at VBM)","vbm"),
                                  ("E_F (zero at Fermi)","efermi"),
                                  ("Absolute (no shift)","absolute")],
                          value="vbm", description="Zero ref")
dpi_int      = w.IntSlider(value=220, min=72, max=600, step=4, description="DPI")
fname_tb     = w.Text(value="bandstructure.png", description="Local file")
render_btn   = w.Button(description="Render plot", button_style="primary")
save_btn     = w.Button(description="Save PNG", button_style="success")
do_remote_cb = w.Checkbox(value=True, description="Also copy to remote bands directory")
msg_out      = w.Output()

_last_fig = {"fig": None}

# ====== helpers ======
CACHE_ROOT = Path.home() / ".atomate2_cache" / "bands"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

def _hash(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]

def _is_remote_like(path_str: str) -> bool:
    """Treat as remote if absolute POSIX path on Windows or path doesn't exist locally."""
    p = path_str.strip()
    if not p:
        return False
    if os.name == "nt" and p.startswith("/"):
        return True
    return not Path(p).exists()

def _rmt():
    R = globals().get("rmt")
    if R is None:
        raise RuntimeError("No 'rmt' object available. Cannot access remote files.")
    return R

def _download_remote_file(remote_path: str, local_path: Path) -> bool:
    """Fetch remote -> local using rmt; uses get()/download() when available, else base64."""
    R = _rmt()
    local_path.parent.mkdir(parents=True, exist_ok=True)
    for attr in ("get", "get_file", "download"):
        if hasattr(R, attr):
            try:
                getattr(R, attr)(remote_path, str(local_path))
                return local_path.exists() and local_path.stat().st_size > 0
            except Exception:
                pass
    cmd = f"set -e; base64 -w 0 {shlex.quote(remote_path)}"
    rc, out, err = R.run(cmd, check=False, modules=False, export_env=False)
    if rc != 0 or not out:
        raise FileNotFoundError(f"Remote read failed for {remote_path}: {err or out}")
    data = base64.b64decode(out.encode("ascii"))
    local_path.write_bytes(data)
    return True

def _ensure_local_inputs(bands_dir: str) -> Path:
    """Ensure local vasprun.xml (or .gz) and KPOINTS; return the local dir path."""
    p_bands = bands_dir.strip()
    if not p_bands:
        raise ValueError("Please set the Bands dir path (folder containing vasprun.xml and KPOINTS).")

    local_vas = Path(p_bands) / "vasprun.xml"
    local_kpt = Path(p_bands) / "KPOINTS"
    if local_vas.exists() and local_kpt.exists():
        return Path(p_bands)

    cache_dir = CACHE_ROOT / _hash(p_bands)
    cache_dir.mkdir(parents=True, exist_ok=True)
    tgt_vas = cache_dir / "vasprun.xml"
    tgt_kpt = cache_dir / "KPOINTS"

    vas_remote = posixpath.join(p_bands, "vasprun.xml")
    vas_gz_remote = posixpath.join(p_bands, "vasprun.xml.gz")
    kpt_remote = posixpath.join(p_bands, "KPOINTS")

    if not tgt_kpt.exists():
        _download_remote_file(kpt_remote, tgt_kpt)

    if not tgt_vas.exists():
        R = _rmt()
        rc_xml, _, _ = R.run(f"test -f {shlex.quote(vas_remote)}", check=False, modules=False, export_env=False)
        rc_gz,  _, _ = R.run(f"test -f {shlex.quote(vas_gz_remote)}", check=False, modules=False, export_env=False)
        if rc_xml == 0:
            _download_remote_file(vas_remote, tgt_vas)
        elif rc_gz == 0:
            gz_path = cache_dir / "vasprun.xml.gz"
            _download_remote_file(vas_gz_remote, gz_path)
            with gzip.open(gz_path, "rb") as fin, open(tgt_vas, "wb") as fout:
                fout.write(fin.read())
        else:
            raise FileNotFoundError(f"Neither vasprun.xml nor vasprun.xml.gz exists in remote dir: {p_bands}")

    if not tgt_vas.exists() or tgt_vas.stat().st_size == 0:
        raise FileNotFoundError(f"Failed to obtain a valid vasprun.xml for {p_bands}")
    if not tgt_kpt.exists() or tgt_kpt.stat().st_size == 0:
        raise FileNotFoundError(f"Failed to obtain a valid KPOINTS for {p_bands}")

    return cache_dir

def _remote_png_path() -> str:
    bdir = bands_dir_tb.value.strip()
    fname = Path(fname_tb.value.strip() or "bandstructure.png").name
    return posixpath.join(bdir, fname) if bdir else ""

def _write_remote_binary(remote_path: str, data: bytes) -> bool:
    """Copy PNG bytes to cluster via base64 using your `rmt` object."""
    try:
        R = _rmt()
    except Exception as e:
        with msg_out: print(f"[warn] {e}")
        return False
    b64 = base64.b64encode(data).decode("ascii")
    tmp = f"/tmp/bands_png_{int(time.time())}_{os.getpid()}.b64"
    cmd = (
        f"set -e; "
        f"mkdir -p {shlex.quote(posixpath.dirname(remote_path))}; "
        f"cat > {shlex.quote(tmp)} <<'B64'\n{b64}\nB64\n"
        f"base64 -d {shlex.quote(tmp)} > {shlex.quote(remote_path)}\n"
        f"rm -f {shlex.quote(tmp)}"
    )
    rc, out, err = R.run(cmd, check=False, modules=False, export_env=False)
    if rc != 0:
        with msg_out: print(f"[remote-copy] failed: {err or out}")
        return False
    return True

def _set_zero_to_vbm(bs):
    try:
        bs.efermi = bs.get_vbm()["energy"]
    except Exception:
        pass
    return bs

def _count_spin_lines(plotter, zero_to_efermi=True):
    """Count number of band lines per spin from bs_plot_data (segments × bands)."""
    data = plotter.bs_plot_data(zero_to_efermi=zero_to_efermi)
    energy = data["energy"]

    def _pick(spin_key):
        if spin_key in energy:
            return energy[spin_key]
        for k in list(energy.keys()):
            if hasattr(spin_key, "value") and (k == spin_key.value or str(k) == str(spin_key.value)):
                return energy[k]
        return None

    e_up = _pick(Spin.up)
    e_dn = _pick(Spin.down)

    def _n_lines(e):
        if not e:
            return 0
        return sum(len(e_seg) for e_seg in e)

    return _n_lines(e_up), _n_lines(e_dn)

def _restyle_spin_lines(ax, plotter, *, zero_to_efermi=True):
    """
    Keep BSPlotter axes; just restyle spin ↓ as dashed **and red**.
    Assumes BSPlotter plots all ↑ first then all ↓ (default behavior).
    """
    up_n, dn_n = _count_spin_lines(plotter, zero_to_efermi=zero_to_efermi)
    if dn_n == 0:
        return  # non-spin-polarized

    # Band lines only (exclude ticks/fermi/separators)
    band_lines = [ln for ln in ax.get_lines() if len(ln.get_xdata()) > 3]
    if not band_lines:
        return

    # Determine which lines correspond to ↓
    if len(band_lines) < up_n + dn_n:
        split = len(band_lines) // 2
        dn_lines = band_lines[split:]
    else:
        dn_start = up_n
        dn_lines = band_lines[dn_start: dn_start + dn_n]

    red_color = "tab:red"
    for ln in dn_lines:
        ln.set_linestyle("--")
        ln.set_color(red_color)      # <-- force red for ↓
        ln.set_linewidth(max(1.0, ln.get_linewidth()))
        ln.set_alpha(0.95)
        ln.set_zorder(3)

    # Legend proxies
    try:
        import matplotlib.lines as mlines
        up_color = band_lines[0].get_color() if band_lines else "tab:blue"
        up_line = mlines.Line2D([], [], color=up_color, linestyle="-", label="spin ↑")
        dn_line = mlines.Line2D([], [], color=red_color, linestyle="--", label="spin ↓")
        ax.legend(handles=[up_line, dn_line], loc="best", frameon=False)
    except Exception:
        pass

def _make_plot():
    # ===== Silence warnings (POTCAR + unconverged) =====
    try:
        from custodian.custodian import UnconvergedVASPWarning as _CUnconv  # type: ignore
    except Exception:
        _CUnconv = None
    try:
        from custodian.vasp.io import UnconvergedVASPWarning as _VUnconv  # type: ignore
    except Exception:
        _VUnconv = None

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module=r"^pymatgen\.io\.vasp\.outputs$")
        if _CUnconv:
            warnings.filterwarnings("ignore", category=_CUnconv)
        if _VUnconv and _VUnconv is not _CUnconv:
            warnings.filterwarnings("ignore", category=_VUnconv)
        warnings.filterwarnings("ignore", message=r".*unconverged VASP run.*", category=Warning)

        # ===== Ensure local inputs =====
        bdir = bands_dir_tb.value.strip()
        local_dir = Path(bdir)
        if _is_remote_like(bdir):
            local_dir = _ensure_local_inputs(bdir)

        vas = local_dir / "vasprun.xml"
        kpt = local_dir / "KPOINTS"
        if not vas.exists():
            raise FileNotFoundError(f"Missing file: {vas}")
        if not kpt.exists():
            raise FileNotFoundError(f"Missing file: {kpt}")

        v = Vasprun(str(vas), parse_projected_eigen=False, parse_potcar_file=False)
        bs = v.get_band_structure(kpoints_filename=str(kpt), line_mode=True, efermi=v.efermi)

    # zero reference
    zr = zero_ref_dd.value
    zero_to_efermi = (zr in ("vbm", "efermi"))
    if zr == "vbm":
        bs = _set_zero_to_vbm(bs)

    # Render with BSPlotter (keep axes), then restyle ↓
    plotter = BSPlotter(bs)
    ax = plotter.get_plot(zero_to_efermi=zero_to_efermi)
    _restyle_spin_lines(ax, plotter, zero_to_efermi=zero_to_efermi)
    fig = ax.figure
    fig.set_dpi(110)
    return fig, local_dir

# ====== actions ======
def _on_render(_):
    with msg_out:
        msg_out.clear_output()
        try:
            fig, local_dir = _make_plot()
            _last_fig["fig"] = fig
            display(Markdown(f"**Preview (not yet saved)**  \nLocal source: `{local_dir}`"))
            display(fig)
            if do_remote_cb.value:
                print(f"On save, will also copy to: {_remote_png_path()}")
            print("Rendered OK. Now set file name and click 'Save PNG'.")
        except Exception as e:
            print("Render error:", e)

def _on_save(_):
    with msg_out:
        try:
            fig = _last_fig["fig"]
            if fig is None:
                print("Nothing to save yet — click 'Render plot' first.")
                return

            # Local save
            local_path = Path(fname_tb.value).expanduser()
            local_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(str(local_path), dpi=int(dpi_int.value), bbox_inches="tight")
            print(f"Saved local PNG → {local_path}")

            # Optional remote copy (to <bands dir>/<local filename>)
            if do_remote_cb.value:
                rpath = _remote_png_path()
                if not rpath:
                    print("[remote-copy] Please set a valid Bands dir first.")
                else:
                    buf = io.BytesIO()
                    fig.savefig(buf, format="png", dpi=int(dpi_int.value), bbox_inches="tight")
                    ok = _write_remote_binary(rpath, buf.getvalue())
                    if ok:
                        print(f"Copied PNG to remote → {rpath}")
        except Exception as e:
            print("Save error:", e)

render_btn.on_click(_on_render)
save_btn.on_click(_on_save)

ui = w.VBox([
    w.HTML("<b>Bandstructure Plotter (manual save, auto remote sync/copy)</b>"),
    bands_dir_tb,
    w.HBox([zero_ref_dd, dpi_int]),
    render_btn,                    # Render button alone
    w.HTML("<b>Save options:</b>"),
    fname_tb,                      # Local file input
    save_btn,                      # Save button under local file
    do_remote_cb,
    msg_out
])
display(ui)


<a id="cell-7"></a>
## 7. Close SSH connection


Closes the persistent SSH session (`rmt`) at the end of the notebook.  
No changes needed—just run this last.

In [22]:
# --- Close persistent Remote session (idempotent & safe) ---
import time, threading

def _safe_call(obj, name, *args, **kwargs):
    try:
        fn = getattr(obj, name, None)
        if callable(fn):
            return fn(*args, **kwargs)
    except Exception:
        pass
    return None

# 1) Stop any monitor polling thread from Cell 4 (best-effort)
try:
    if isinstance(globals().get("_stop_event"), threading.Event):
        globals()["_stop_event"].set()
except Exception:
    pass

# 2) Close SSH port-forward tunnels (from Cell 1)
SSH_SESSION = globals().get("SSH_SESSION") or {}
tunnels = []
try:
    tunnels = list(SSH_SESSION.get("tunnels", []))
except Exception:
    pass

stopped_tunnels = 0
for t in tunnels:
    try:
        _safe_call(t.get("stop_event"), "set")
        _safe_call(t.get("server"), "server_close")
        thr = t.get("thread")
        if isinstance(thr, threading.Thread) and thr.is_alive():
            try:
                thr.join(timeout=1.5)
            except Exception:
                pass
        stopped_tunnels += 1
    except Exception:
        pass

# 3) Close the Remote wrapper and underlying Paramiko client
r = globals().get("rmt")
if r:
    # Prefer context-manager exit if provided
    if _safe_call(r, "__exit__", None, None, None) is None:
        _safe_call(r, "__exit__")  # alternate signature
    # Common fallbacks on the wrapper
    for m in ("close", "shutdown", "disconnect", "stop"):
        _safe_call(r, m)

# Close the underlying paramiko client if reachable
client = None
try:
    client = SSH_SESSION.get("client") or getattr(r, "client", None)
except Exception:
    client = getattr(r, "client", None)

_safe_call(client, "close")

# 4) Mark session disconnected and clear references
try:
    SSH_SESSION["connected"] = False
    SSH_SESSION["client"] = None
    SSH_SESSION["tunnels"] = []
except Exception:
    pass

try:
    del globals()["rmt"]
except Exception:
    globals()["rmt"] = None

# 5) Update the status badge in Cell 1 (if present)
status_html = globals().get("status_html")
try:
    if status_html is not None:
        status_html.value = "<b style='color:red'>🔴 Not connected</b>"
except Exception:
    pass

# 6) Nice summary
host = getattr(globals().get("cfg", object()), "username", None)
print(f"Remote session closed. Tunnels stopped: {stopped_tunnels}{' (user='+str(host)+')' if host else ''}")


Remote session closed. Tunnels stopped: 1 (user=leeburton)
